# E23 — six levers on 4×L4 (vLLM 0.32, Qwen3.8-27B), target 1500+ tok/s · rev 3 · SMOKE_A

The user's six experiments, each with only its own test sequence (DESIGN.md), at the production context
(thinking up to 120k: `--max-model-len 131072`, bands 2k / 8k / 40k / 90k / 120k):
1. TP1×4 / PP4 (Merlin W4A16 TP1 replicas; PP4 branch on stock vLLM + RedHat INT4, eager + compiled)
2. Four-rank host-staged all-reduce (P5 bench)
3. FULL-graph speculative verification (G0 FlashInfer PIECEWISE, G1/G2 TRITON FULL, G3 FlashInfer + #60532) with a graph verdict
4. SM89 FP8 GEMM (#61224 CUTLASS vs Marlin, P4 bench) + an INT4 TP4 control row
5. NVFP4 KV: correctness (greedy + teacher-forced 40k / 90k), 90k / 120k residency, + MTP3
6. MTP OFF / K1 / K3 at 8/12/16/20 active streams (step time from the iteration histogram)
plus EAGLE-4 feasibility (SMOKE B; FULL only with E23_EAGLE_FULL=1).

This copy runs `MODE=SMOKE`, `PART=A` by default (the code is identical in the three E23 notebooks).
Every number is a measured counter; a missing measurement is INCONCLUSIVE / SKIPPED, never PASS or zero.


In [ ]:
# ==================================================================== CELL 1 — config, inputs, lab module
import os, sys, json, time, re, shutil, subprocess, hashlib, tarfile, threading, traceback
from pathlib import Path
T_START = time.time()
MODE = os.environ.get("Q27X_E23_MODE", "SMOKE").upper()        # SMOKE | FULL (baked per notebook)
PART = os.environ.get("Q27X_E23_PART", "A").upper()        # SMOKE: A (Merlin) | B (stock + EAGLE) | ALL
assert MODE in ("SMOKE", "FULL") and PART in ("A", "B", "ALL"), (MODE, PART)
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
FAKE = os.environ.get("E23_FAKE_SERVER")                          # local CPU harness only
BUDGET = int(os.environ.get("E23_BUDGET_SEC") or {"SMOKE": {"A": 120, "B": 150, "ALL": 250}[PART] * 60, "FULL": 690 * 60}[MODE])
HARD_STOP = T_START + BUDGET
MAX_LEN = 131072                                                  # user 2026-10-29: thinking up to 120k
OUT = Path(os.environ.get("E23_OUT", "/kaggle/working/e23"))
OUT.mkdir(parents=True, exist_ok=True)
LOGS = OUT / "logs"
LOGS.mkdir(exist_ok=True)
TMP = Path(os.environ.get("E23_TMP", "/tmp"))
INP = Path(os.environ.get("E23_INPUT", "/kaggle/input"))
R = {"mode": MODE, "part": PART, "budget_min": BUDGET // 60, "max_len": MAX_LEN, "rev": "3",
     "start_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "setup": {}, "sections": {}, "errors": [], "order": []}
_SAVE_LOCK = threading.Lock()


def save():
    with _SAVE_LOCK:
        tmp = OUT / "e23_results.json.tmp"
        tmp.write_text(json.dumps(R, indent=1, default=str))
        tmp.replace(OUT / "e23_results.json")


def left():
    return HARD_STOP - time.time()


def log(*a):
    print(f"[{(time.time() - T_START) / 60:6.1f} min]", *a, flush=True)


def find(pattern, root=INP, must=True):
    hits = sorted(Path(root).rglob(pattern)) if Path(root).exists() else []
    if not hits and must:
        raise FileNotFoundError(f"{pattern} not under {root}: attach the input")
    return hits[0] if hits else None


if RERUN:      # rev 3: a competition rerun writes the minimal valid submission and skips cells 2-6 (no exception)
    tasks = json.load(open(find("arc-agi_test_challenges.json")))
    sub = {k: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in v["test"]] for k, v in tasks.items()}
    Path("/kaggle/working/submission.json").write_text(json.dumps(sub))
    print("E23 lab notebook: competition rerun -> fallback submission.json only")
else:
    LAB_SRC = '"""E23 lab, rev 3: port-parameterized server lifecycle and measurement for the six-lever notebook (stdlib only).\n\nBase: the Exp 14 VMS harness (VServer / VGen: token ids from return_token_ids, [DONE] + finish_reason required, socket\nshutdown on stop, top-1/top-2 margins, garbage checks on every window). Adopted from the consolidated v3 notebook:\nsame-weight greedy oracle classified by the oracle\'s logprob margin at the first differing token, a steady decode window\nthat starts only after prefill settled, a GPU-release gate, fail-closed statuses, checkpoint identity, a runtime\n--help flag check, KV + GPU memory samples.\nrev 3 (validation agents 2026-10-29, measurement + robustness):\n  - port check by connect (a TIME_WAIT socket of the previous server is not "busy"), with a 90 s wait;\n  - kill(): SIGKILL the whole session after SIGTERM, even when the leader already exited;\n  - steady(): idle gate at entry, admission cap from the prefill to do (700 tok/s assumed), settle needs no preemption\n    and every running stream gaining tokens; in the window: stalled streams, a changing running/waiting set (KV-bound),\n    preemptions and prefills make it INCONCLUSIVE (numbers kept); garbage under load fails it; steps from the\n    vllm:iteration_tokens_total histogram count; replicas measured in one shared window (barrier);\n  - profiled_load(): settle as steady(), bounded profile (delay / max iterations), with_stack off, frontend ignored;\n  - trace_summary(): categories on the kernel base name (attention before GDN: "fla" is inside "flashinfer"), per-forward\n    graph / kernel launch counts inside the gpu_model_runner forward ranges (VLLM_CUSTOM_SCOPES_FOR_PROFILING=1) and the\n    CUPTI correlation fallback -> FULL / PIECEWISE / EAGER / UNKNOWN; startup cudagraph lines parsed explicitly;\n  - teacher_forced(): prompt logprobs of the last N positions of a long prompt (NVFP4 vs FP8 KV at 40k / 90k);\n  - logprobs keyed by token id (return_tokens_as_token_ids); stop() race closed; compare() flags garbage only when the\n    oracle itself is clean.\n"""\nimport gzip\nimport hashlib\nimport json\nimport os\nimport re\nimport signal\nimport socket\nimport statistics\nimport subprocess\nimport threading\nimport time\nimport urllib.request as U\nfrom pathlib import Path\n\nFAIL_MARGIN = 0.5          # first differing token: oracle top-1 minus top-2 logprob at or above this = real divergence\nLIVE = set()               # every running Server, for the hard-stop watchdog\n\n\n# ------------------------------------------------------------------ server lifecycle\ndef port_listening(port):\n    """True when something accepts connections on 127.0.0.1:port (TIME_WAIT leftovers do not)."""\n    s = socket.socket()\n    s.settimeout(1.0)\n    try:\n        s.connect(("127.0.0.1", int(port)))\n        return True\n    except OSError:\n        return False\n    finally:\n        s.close()\n\n\nclass Server:\n    def __init__(self, name, argv, env, port, log_dir, gpus="0,1,2,3"):\n        self.name, self.argv, self.env, self.port = name, [str(a) for a in argv], dict(env), int(port)\n        self.gpus = str(gpus)\n        self.log_path = Path(log_dir) / f"server_{name}.log"\n        self.proc, self.t0, self.ready_sec = None, None, None\n\n    @property\n    def url(self):\n        return f"http://127.0.0.1:{self.port}"\n\n    def start(self):\n        end = time.time() + 90\n        while port_listening(self.port):\n            if time.time() > end:\n                raise RuntimeError(f"PORT_BUSY {self.port} (a listener is still there)")\n            time.sleep(3)\n        self.log_path.parent.mkdir(parents=True, exist_ok=True)\n        self._fh = open(self.log_path, "w", encoding="utf-8")\n        self.t0 = time.perf_counter()\n        self.proc = subprocess.Popen(self.argv, env=self.env, stdout=self._fh, stderr=subprocess.STDOUT,\n                                     start_new_session=True, cwd="/tmp" if os.path.isdir("/tmp") else None)\n        LIVE.add(self)\n        return self\n\n    def alive(self):\n        return self.proc is not None and self.proc.poll() is None\n\n    def log(self):\n        try:\n            return self.log_path.read_text(encoding="utf-8", errors="replace")\n        except Exception:\n            return ""\n\n    def rss_mb(self):\n        """Host RAM of the server\'s whole session (engine + workers), MB; None off Linux."""\n        try:\n            r = subprocess.run(["ps", "-o", "rss=", "--sid", str(self.proc.pid)], capture_output=True, text=True, timeout=20)\n            return round(sum(int(x) for x in r.stdout.split() if x.isdigit()) / 1024, 1)\n        except Exception:\n            return None\n\n    def wait_ready(self, timeout):\n        while time.perf_counter() - self.t0 < timeout:\n            if not self.alive():\n                return False, f"exited rc={self.proc.returncode}"\n            try:\n                U.urlopen(self.url + "/health", timeout=10).read()\n                self.ready_sec = round(time.perf_counter() - self.t0, 1)\n                return True, f"ready in {self.ready_sec}s"\n            except Exception:\n                time.sleep(3)\n        return False, "startup timeout"\n\n    def kill(self):\n        if self.proc is None:\n            return\n        pg = self.proc.pid\n        try:\n            os.killpg(pg, signal.SIGTERM) if hasattr(os, "killpg") else self.proc.terminate()\n        except Exception:\n            pass\n        try:\n            self.proc.wait(timeout=40)\n        except Exception:\n            pass\n        # rev 3: the leader may exit while EngineCore / workers of the session still hold GPUs: always SIGKILL the group\n        try:\n            os.killpg(pg, signal.SIGKILL) if hasattr(os, "killpg") else self.proc.kill()\n        except Exception:\n            pass\n        try:\n            self.proc.wait(timeout=30)\n        except Exception:\n            pass\n        try:\n            self._fh.close()\n        except Exception:\n            pass\n        LIVE.discard(self)\n\n    def facts(self):\n        """What vLLM logged at startup: KV pool, max concurrency, selected kernels, graph mode, backend, KV dtype."""\n        t = self.log()\n        kv = [int(x.replace(",", "")) for x in re.findall(r"GPU KV cache size: ([\\d,]+) tokens", t)]\n        mc = re.findall(r"Maximum concurrency for [\\d,]+ tokens per request: ([\\d.]+)x", t)\n        kern = sorted(set(re.findall(r"Selected (\\w+Kernel) for", t)))\n        errs = [l[:300] for l in t.splitlines() if re.search(r"Traceback|CUDA error|illegal memory|out of memory|"\n                                                               r"RuntimeError|AssertionError", l)][:12]\n        pick = lambda pat, n=8: [l.strip()[:240] for l in t.splitlines() if re.search(pat, l, re.I)][:n]\n        # rev 3 (measurement H5): the decisive startup evidence for the graph mode actually used with spec decode\n        downgrade = pick(r"not supported with spec-decode.*setting cudagraph_mode|setting cudagraph_mode\\s*=", 4)\n        capture = sorted(set(re.findall(r"Capturing CUDA graphs \\(([^)]+)\\)", t)))\n        configured = sorted(set(re.findall(r"CUDAGraphMode\\.(\\w+)", t)))\n        return {"kv_tokens": min(kv) if kv else None, "kv_tokens_all": kv, "max_concurrency": float(mc[-1]) if mc else None,\n                "kernels": kern, "error_lines": errs, "ready_sec": self.ready_sec,\n                "graph_capture_modes": capture, "graph_downgrade_lines": downgrade, "cudagraph_modes_logged": configured,\n                "backend_lines": pick(r"attention backend|Using \\w+ backend|backend=|AttentionBackend"),\n                "kv_dtype_lines": pick(r"kv_cache_dtype|kv cache dtype|nvfp4"),\n                "host_embed_lines": pick(r"host[- _]?(resident|embed)|HOST_EMBED|embed.*(cpu|host)", 6)}\n\n\ndef gpu_pids(gpus=None):\n    """Compute PIDs on all GPUs, or only on the GPU indices in `gpus` (a bench on another GPU must not fail the release\n    gate of a server that used GPU 0 only). None when nvidia-smi is unavailable."""\n    try:\n        a = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,gpu_uuid", "--format=csv,noheader,nounits"],\n                           capture_output=True, text=True, timeout=30)\n        g = subprocess.run(["nvidia-smi", "--query-gpu=index,uuid", "--format=csv,noheader,nounits"],\n                           capture_output=True, text=True, timeout=30)\n    except Exception:\n        return None\n    if a.returncode or g.returncode:\n        return None\n    idx = {u.strip(): int(i) for i, u in (l.split(",", 1) for l in g.stdout.strip().splitlines() if "," in l)}\n    want = None if gpus is None else {int(x) for x in str(gpus).split(",") if str(x).strip() != ""}\n    out = set()\n    for l in a.stdout.strip().splitlines():\n        if "," not in l:\n            continue\n        pid, uuid = (x.strip() for x in l.split(",", 1))\n        if pid.isdigit() and (want is None or idx.get(uuid) in want):\n            out.add(int(pid))\n    return sorted(out)\n\n\ndef gpu_mem():\n    try:\n        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],\n                           capture_output=True, text=True, timeout=30)\n        return [[int(v) for v in l.split(",")] for l in r.stdout.strip().splitlines() if l.strip()]\n    except Exception:\n        return None\n\n\ndef wait_gpu_release(timeout=180, fake=False, gpus=None, free_mib=1024):\n    """True when the GPUs in `gpus` (all when None) hold no compute PID AND use < free_mib MiB each (rev 3: compute-app\n    PIDs are often invisible inside Kaggle containers; memory.used is the proven E18 gate); None without nvidia-smi."""\n    if fake:\n        return True\n    want = None if gpus is None else {int(x) for x in str(gpus).split(",") if str(x).strip() != ""}\n    end = time.time() + timeout\n    while time.time() < end:\n        p, mem = gpu_pids(gpus), gpu_mem()\n        if p is None or mem is None:\n            return None\n        used = [m[1] for m in mem if want is None or m[0] in want]\n        if not p and all(u < free_mib for u in used):\n            return True\n        time.sleep(3)\n    return False\n\n\ndef identity(model_dir):\n    """Checkpoint identity: sha256 of config / tokenizer / index files (an oracle is only compared with the same weights)."""\n    root = Path(model_dir)\n    files = [p for n in ("config.json", "tokenizer.json", "tokenizer_config.json", "generation_config.json")\n             for p in [root / n] if p.is_file()] + sorted(root.glob("*.safetensors.index.json"))\n    h = hashlib.sha256()\n    for p in files:\n        h.update(p.name.encode())\n        h.update(hashlib.sha256(p.read_bytes()).digest())\n    return {"dir": str(root), "files": [p.name for p in files], "sha256": h.hexdigest()[:16]}\n\n\n_HELP = {}\n\n\ndef help_check(key, argv_prefix, env, flags, timeout=300):\n    """The engine\'s own `serve --help=all` (run once per engine `key`, cached) must list every flag of THIS launch\n    (rev 3: every launch is checked, not only the first); returns the missing ones."""\n    if key not in _HELP:\n        try:\n            r = subprocess.run(list(argv_prefix) + ["--help=all"], capture_output=True, text=True, timeout=timeout, env=env)\n            _HELP[key] = r.stdout + r.stderr\n        except Exception as e:\n            _HELP[key] = "ERROR " + repr(e)[:200]\n    txt = _HELP[key]\n    if len(txt) < 2000:\n        return {"ran": False, "error": "help output too short or failed", "tail": txt[-400:]}\n    want = sorted({f for f in flags if isinstance(f, str) and f.startswith("--")})\n    return {"ran": True, "missing": [f for f in want if f not in txt]}\n\n\ndef start_watchdog(hard_stop_epoch, on_fire):\n    """A real hard stop: at `hard_stop_epoch` every live Server is killed and on_fire() is called."""\n    def run():\n        while time.time() < hard_stop_epoch:\n            time.sleep(5)\n        try:\n            on_fire()\n        finally:\n            for s in list(LIVE):\n                try:\n                    s.kill()\n                except Exception:\n                    pass\n    t = threading.Thread(target=run, daemon=True, name="e23-watchdog")\n    t.start()\n    return t\n\n\ndef prefetch(paths, threads=16, chunk=16 << 20):\n    """Read files once with many readers (page cache warm-up of a cold /kaggle/input mount); returns bytes and seconds."""\n    files = [p for root in paths for p in sorted(Path(root).rglob("*")) if p.is_file()]\n    t0, done, lock = time.time(), [0], threading.Lock()\n\n    def worker(fs):\n        for p in fs:\n            try:\n                with open(p, "rb") as f:\n                    while True:\n                        b = f.read(chunk)\n                        if not b:\n                            break\n                        with lock:\n                            done[0] += len(b)\n            except Exception:\n                pass\n    big = sorted(files, key=lambda p: -p.stat().st_size)\n    parts = [big[i::threads] for i in range(threads)]\n    ths = [threading.Thread(target=worker, args=(x,), daemon=True) for x in parts if x]\n    for t in ths:\n        t.start()\n    for t in ths:\n        t.join()\n    dt = time.time() - t0\n    return {"files": len(files), "bytes": done[0], "sec": round(dt, 1), "MBps": round(done[0] / 1e6 / max(dt, 1e-3), 1)}\n\n\n# ------------------------------------------------------------------ metrics\ndef prom(port):\n    txt = U.urlopen(f"http://127.0.0.1:{port}/metrics", timeout=15).read().decode("utf-8", "replace")\n    out = {}\n    for line in txt.splitlines():\n        if not line or line.startswith("#"):\n            continue\n        m = re.match(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\\{[^}]*\\})?\\s+(\\S+)", line)\n        if not m:\n            continue\n        try:\n            v = float(m.group(3))\n        except ValueError:\n            continue\n        if v != v:\n            continue\n        out[m.group(1)] = out.get(m.group(1), 0.0) + v\n        out[m.group(1) + "@max"] = max(out.get(m.group(1) + "@max", v), v)\n    return out\n\n\nGEN, PROMPT, RUN, WAIT = ("vllm:generation_tokens_total", "vllm:prompt_tokens_total", "vllm:num_requests_running",\n                          "vllm:num_requests_waiting")\nPREEMPT, KVU, ITER = "vllm:num_preemptions_total", "vllm:kv_cache_usage_perc", "vllm:iteration_tokens_total_count"\nACC, DRF, DTOK = ("vllm:spec_decode_num_accepted_tokens_total", "vllm:spec_decode_num_drafts_total",\n                  "vllm:spec_decode_num_draft_tokens_total")\n\n\n# ------------------------------------------------------------------ one streaming request (Exp 14 VGen, port-aware)\nclass Stream(threading.Thread):\n    def __init__(self, port, ids, max_new, temperature=1.0, seed=None, logprobs=0, ignore_eos=True, model="q27"):\n        super().__init__(daemon=True)\n        self.port = int(port)\n        self.body = {"model": model, "prompt": list(ids), "max_tokens": int(max_new), "temperature": temperature,\n                     "top_p": 0.95 if temperature else 1.0, "stream": True, "ignore_eos": ignore_eos, "return_token_ids": True}\n        if temperature:\n            self.body["top_k"] = 20\n        if seed is not None:\n            self.body["seed"] = int(seed)\n        if logprobs:\n            self.body["logprobs"] = int(logprobs)\n            self.body["return_tokens_as_token_ids"] = True      # rev 3: logprob keys by id (no string collisions)\n        self.text, self.tokens, self.margins = [], [], []\n        self.err, self.done_seen, self.finish_reason = None, False, None\n        self.t_start = self.t_first = self.t_end = None\n        self._resp, self._halt = None, False\n\n    def run(self):\n        self.t_start = time.perf_counter()\n        try:\n            req = U.Request(f"http://127.0.0.1:{self.port}/v1/completions", data=json.dumps(self.body).encode(),\n                            method="POST", headers={"Content-Type": "application/json"})\n            self._resp = U.urlopen(req, timeout=7200)\n            for raw in self._resp:\n                if self._halt:\n                    break\n                line = raw.strip()\n                if not line.startswith(b"data:"):\n                    continue\n                payload = line[5:].strip()\n                if payload == b"[DONE]":\n                    self.done_seen = True\n                    break\n                d = json.loads(payload)\n                if "error" in d or d.get("object") == "error":\n                    self.err = ("stream error: " + json.dumps(d.get("error") or d.get("message") or d))[:300]\n                    break\n                for c in d.get("choices") or []:\n                    self.text.append(c.get("text") or "")\n                    ids_ = c.get("token_ids")\n                    lp = c.get("logprobs") or {}\n                    if ids_ is None:\n                        if c.get("text") or lp.get("tokens"):\n                            self.err = "MISSING_STREAM_TOKEN_IDS"\n                    elif any(not isinstance(v, int) or isinstance(v, bool) for v in ids_):\n                        self.err = "INVALID_STREAM_TOKEN_IDS"\n                    else:\n                        if ids_ and self.t_first is None:\n                            self.t_first = time.perf_counter()\n                        self.tokens.extend(ids_)\n                    for top in lp.get("top_logprobs") or []:\n                        vals = sorted((v for v in (top or {}).values() if isinstance(v, (int, float))), reverse=True)\n                        self.margins.append(round(vals[0] - vals[1], 4) if len(vals) >= 2 else None)\n                    if c.get("finish_reason") is not None:\n                        self.finish_reason = c["finish_reason"]\n                if self.err:\n                    break\n        except Exception as e:\n            if not self._halt:\n                self.err = repr(e)[:300]\n        finally:                         # rev 3: a stop() before the response existed still closes the socket here\n            try:\n                if self._resp is not None:\n                    self._resp.close()\n            except Exception:\n                pass\n        self.t_end = time.perf_counter()\n        if not self._halt and not self.err and not (self.done_seen and self.finish_reason):\n            self.err = f"INCOMPLETE_SSE done={self.done_seen} finish={self.finish_reason}"\n\n    def stop(self):\n        self._halt = True\n        try:\n            self._resp.fp.raw._sock.shutdown(socket.SHUT_RDWR)\n        except Exception:\n            pass\n        try:\n            if self._resp is not None:\n                self._resp.close()\n        except Exception:\n            pass\n\n    def row(self):\n        return {"text": "".join(self.text), "tokens": list(self.tokens), "margins": list(self.margins), "error": self.err,\n                "done": self.done_seen, "finish_reason": self.finish_reason}\n\n\n# ------------------------------------------------------------------ correctness\ndef greedy(port, prompts, n_tok=128, logprobs=2, timeout=900, model="q27"):\n    """Greedy rows for the same-weight oracle (one request at a time: batch-size effects stay out of the comparison)."""\n    rows = []\n    for ids in prompts:\n        g = Stream(port, ids, n_tok, temperature=0.0, logprobs=logprobs, ignore_eos=True, model=model)\n        g.start()\n        g.join(timeout)\n        if g.is_alive():\n            g.stop()\n            g.join(10)\n            r = g.row()\n            r["error"] = r["error"] or "GREEDY_TIMEOUT"\n        else:\n            r = g.row()\n        rows.append(r)\n    return rows\n\n\ndef degenerate(text):\n    """Objective garbage checks: empty, a run of one char, \'!!!!\' (the E3 / E13 garbage), very low variety."""\n    t = text or ""\n    if not t.strip():\n        return "EMPTY"\n    if re.search(r"(\\S)\\1{200,}", t, re.S):      # rev 3: grid rows / separators are not garbage\n        return "REPEATED_CHAR"\n    if t.count("!") > 0.5 * max(1, len(t)):\n        return "BANG_GARBAGE"\n    words = re.findall(r"\\w+", t)\n    if len(words) > 40 and len(set(words)) < 0.08 * len(words):\n        return "LOW_VARIETY"\n    return None\n\n\ndef compare(ref, cand):\n    """Same-weight oracle: at the first differing token the ORACLE\'s top-1 - top-2 logprob margin decides:\n    >= FAIL_MARGIN -> FAIL_HIGH_MARGIN, below -> NEAR_TIE; identical -> EXACT; missing data -> INCONCLUSIVE."""\n    if not ref or not cand or len(ref) != len(cand):\n        return {"status": "INCONCLUSIVE_PAIRS_MISSING"}\n    per, worst = [], "EXACT"\n    rank = {"EXACT": 0, "NEAR_TIE": 1, "INCONCLUSIVE": 2, "FAIL_HIGH_MARGIN": 3}\n    for a, b in zip(ref, cand):\n        if a.get("error") or b.get("error") or not a.get("tokens") or not b.get("tokens"):\n            st, d, m = "INCONCLUSIVE", None, None\n        else:\n            x, y = a["tokens"], b["tokens"]\n            d = next((i for i in range(min(len(x), len(y))) if x[i] != y[i]), None)\n            if d is None:\n                st, m = ("EXACT" if len(x) == len(y) else "INCONCLUSIVE"), None\n            else:\n                mm = a.get("margins") or []\n                m = mm[d] if d < len(mm) else None\n                st = "INCONCLUSIVE" if m is None else ("FAIL_HIGH_MARGIN" if m >= FAIL_MARGIN else "NEAR_TIE")\n        g = degenerate(b.get("text"))\n        if g and degenerate(a.get("text")) is None:           # rev 3: only when the oracle itself is clean\n            st = "FAIL_HIGH_MARGIN"\n        per.append({"first_diff": d, "oracle_margin": m, "status": st, "garbage": g})\n        worst = st if rank[st] > rank[worst] else worst\n    return {"status": worst, "rows": per}\n\n\ndef chat_probes(port, model="q27", cases=(("Reply with exactly one number: 7 * 6 =", "42"),\n                                          ("Reply with exactly one number: 2 + 2 =", "4"),\n                                          ("Reply with exactly one number: 9 - 3 =", "6")), timeout=300):\n    """Arithmetic through /v1/chat/completions with thinking off. vLLM 0.32 returns the answer in message.content and\n    the thinking in message.reasoning (NOT reasoning_content: the consolidated notebook\'s blocker)."""\n    out = []\n    for q, want in cases:\n        body = {"model": model, "messages": [{"role": "user", "content": q}], "temperature": 0, "max_tokens": 32,\n                "chat_template_kwargs": {"enable_thinking": False}}\n        try:\n            r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=json.dumps(body).encode(),\n                                               headers={"Content-Type": "application/json"}), timeout=timeout).read())\n            msg = r["choices"][0]["message"]\n            got = (msg.get("content") or "").strip()\n            out.append({"q": q, "want": want, "content": got[:80], "reasoning_chars": len(msg.get("reasoning") or ""),\n                        "ok": re.search(r"(?<!\\d)" + want + r"(?!\\d)", got) is not None})\n        except Exception as e:\n            out.append({"q": q, "want": want, "error": repr(e)[:200], "ok": False})\n    return {"ok": all(x["ok"] for x in out), "cases": out}\n\n\ndef teacher_forced(port, ids, last=256, model="q27", timeout=1800):\n    """rev 3 (measurement M3): prompt logprobs of the last `last` positions of a long prompt (one prefill, 1 token out),\n    for a lossy-KV comparison (NVFP4 vs FP8 KV) at the context lengths that matter. Returns per-position\n    (logprob of the actual token, rank) or an error."""\n    body = {"model": model, "prompt": list(ids), "max_tokens": 1, "temperature": 0, "prompt_logprobs": 1,\n            "return_tokens_as_token_ids": True}\n    try:\n        r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/v1/completions", data=json.dumps(body).encode(),\n                                           headers={"Content-Type": "application/json"}), timeout=timeout).read())\n        pl = r["choices"][0].get("prompt_logprobs") or []\n    except Exception as e:\n        return {"error": repr(e)[:300]}\n    out = []\n    for pos in range(max(1, len(ids) - last), len(ids)):\n        d = pl[pos] if pos < len(pl) else None\n        if not isinstance(d, dict):\n            out.append(None)\n            continue\n        e = d.get(str(ids[pos])) or d.get(f"token_id:{ids[pos]}") or next(iter(d.values()), None)\n        out.append([e.get("logprob"), e.get("rank")] if isinstance(e, dict) else None)\n    return {"positions": out, "n": len(out)}\n\n\ndef compare_tf(ref, cand):\n    """Teacher-forced comparison: mean / max |delta logprob| of the actual tokens and rank-1 agreement."""\n    if not ref or not cand or ref.get("error") or cand.get("error"):\n        return {"status": "INCONCLUSIVE"}\n    pairs = [(a, b) for a, b in zip(ref["positions"], cand["positions"]) if a and b and a[0] is not None and b[0] is not None]\n    if not pairs:\n        return {"status": "INCONCLUSIVE"}\n    d = [abs(a[0] - b[0]) for a, b in pairs]\n    r1 = sum(1 for a, b in pairs if (a[1] == 1) == (b[1] == 1)) / len(pairs)\n    return {"status": "MEASURED", "n": len(pairs), "mean_abs_dlogprob": round(statistics.mean(d), 4),\n            "max_abs_dlogprob": round(max(d), 3), "rank1_agreement": round(r1, 4)}\n\n\n# ------------------------------------------------------------------ steady decode\ndef steady(port, prompts, level, secs=60.0, admit=None, max_new=8192, temperature=1.0, seed=1234, model="q27",\n           sample=2.0, prefill_tok_s=700.0, gpu_sample=True, barrier=None):\n    """`level` distinct prompts, then a decode window that starts only once prefill has settled. Status:\n    STEADY / CAPACITY_LIMITED (fewer running than asked: KV) / INCONCLUSIVE_* / FAIL_*; numbers are kept on an\n    INCONCLUSIVE window so a KV census survives; never a made-up number. `barrier` (steady_multi) starts every replica\'s\n    window at the same moment."""\n    plen = len(prompts[0]) if prompts else 0\n    admit = admit or max(420.0, 120.0 + level * plen / prefill_tok_s)\n    r = {"level": level, "status": "INCONCLUSIVE_NOT_STARTED", "port": port, "prompt_tokens": plen, "max_new": max_new,\n         "admit_limit_sec": round(admit)}\n    if not wait_idle(port, 60):\n        r["status"] = "INCONCLUSIVE_NOT_IDLE_AT_START"\n        if barrier:\n            barrier.abort()\n        return r\n    try:\n        p0 = prom(port)\n    except Exception as e:\n        r.update(status="INCONCLUSIVE_METRICS_UNAVAILABLE", error=repr(e)[:200])\n        if barrier:\n            barrier.abort()\n        return r\n    if GEN not in p0 or PROMPT not in p0:\n        r["status"] = "INCONCLUSIVE_COUNTERS_MISSING"\n        if barrier:\n            barrier.abort()\n        return r\n    ss = [Stream(port, prompts[i % len(prompts)], max_new, temperature, seed + i, model=model) for i in range(level)]\n    for g in ss:\n        g.start()\n    try:\n        t_adm, best, settled, prev, samples, flat_since = time.perf_counter(), 0, 0, None, [], None\n        last_tok = [0] * len(ss)\n        while time.perf_counter() - t_adm < admit:\n            time.sleep(sample)\n            if all(not g.is_alive() for g in ss):          # rev 3: checked before the metrics read (a dead server)\n                r.update(status="INCONCLUSIVE_STREAMS_ENDED_IN_ADMISSION", errors=[g.err for g in ss if g.err][:3])\n                return r\n            try:\n                m = prom(port)\n            except Exception:\n                continue\n            run, wait, pt, pre = int(m.get(RUN, 0)), int(m.get(WAIT, 0)), m.get(PROMPT), m.get(PREEMPT, 0)\n            now_tok = [len(g.tokens) for g in ss]\n            first = sum(g.t_first is not None for g in ss)\n            gaining = sum(1 for a, b in zip(last_tok, now_tok) if b > a)\n            last_tok = now_tok\n            best = max(best, run)\n            samples.append([round(time.perf_counter() - t_adm, 1), run, wait, first, gaining])\n            # rev 3: no preemption in the sample, every running request emitted and is gaining tokens\n            calm = run > 0 and prev is not None and pt == prev[0] and pre == prev[1] and first >= run and gaining >= run\n            if calm and run >= level and wait == 0:\n                settled += 1\n            elif calm and run < level and wait > 0:       # KV-bound: the rest wait; accept a plateau held 30 s\n                flat_since = flat_since or time.perf_counter()\n                settled = settled + 1 if time.perf_counter() - flat_since > 30 else 0\n            else:\n                settled, flat_since = 0, None\n            prev = (pt, pre)\n            if settled >= 3:\n                break\n        r["admission"] = {"sec": round(time.perf_counter() - t_adm, 1), "best_running": best, "tail": samples[-6:],\n                          "prefill_tok_s": round((prom(port).get(PROMPT, 0) - p0.get(PROMPT, 0)) /\n                                                 max(1e-3, time.perf_counter() - t_adm), 1)}\n        if settled < 3:\n            r["status"] = "INCONCLUSIVE_NO_SETTLE"\n            if barrier:\n                barrier.abort()\n            return r\n        if barrier:\n            try:\n                barrier.wait(timeout=max(60.0, admit))\n            except threading.BrokenBarrierError:\n                r["status"] = "INCONCLUSIVE_REPLICA_BARRIER"\n                return r\n        m0, t0 = prom(port), time.perf_counter()\n        runs, waits, kvs, mids, gmem, stalls = [], [], [], [], [], [0] * len(ss)\n        last_tok = [len(g.tokens) for g in ss]\n        while time.perf_counter() - t0 < secs:\n            time.sleep(min(sample, max(0.1, secs - (time.perf_counter() - t0))))\n            m = prom(port)\n            mids.append(m)\n            runs.append(m.get(RUN, 0))\n            waits.append(m.get(WAIT, 0))\n            if KVU in m:\n                kvs.append(m[KVU])\n            now_tok = [len(g.tokens) for g in ss]\n            for i, g in enumerate(ss):\n                if g.t_first is not None and g.is_alive():\n                    stalls[i] = stalls[i] + 1 if now_tok[i] <= last_tok[i] else 0\n            last_tok = now_tok\n            if any(x >= 3 for x in stalls):\n                break\n            if gpu_sample and len(mids) % 5 == 1:\n                g_ = gpu_mem()\n                if g_:\n                    gmem.append(g_)\n        m1, t1 = prom(port), time.perf_counter()\n        gen = m1.get(GEN, 0) - m0.get(GEN, 0)\n        dt = t1 - t0\n        acc, drf, dtk, it = (m1.get(k, 0) - m0.get(k, 0) for k in (ACC, DRF, DTOK, ITER))\n        rm = statistics.mean(runs) if runs else 0\n        steps = it if it else ((drf / rm) if (drf and rm) else ((gen / rm) if rm else 0))\n        r.update(tok_s=round(gen / dt, 1) if dt else None, gen_tokens=int(gen), window_sec=round(dt, 1),\n                 running_mean=round(rm, 2), running_min=min(runs or [0]), running_max=max(runs or [0]),\n                 waiting_min=min(waits or [0]), waiting_max=max(waits or [0]),\n                 tok_s_per_stream=round(gen / dt / rm, 2) if rm and dt else None,\n                 accept_per_draft=round(acc / drf, 3) if drf else None,\n                 mean_acceptance_length=round(1 + acc / drf, 3) if drf else None,\n                 accept_rate=round(acc / dtk, 3) if dtk else None,\n                 steps=int(steps), steps_source="iteration_histogram" if it else "estimate",\n                 step_ms=round(dt * 1000 / steps, 1) if steps else None,\n                 tokens_per_step=round(gen / steps, 2) if steps else None,\n                 kv_usage_mean=round(statistics.mean(kvs), 3) if kvs else None, kv_usage_max=max(kvs) if kvs else None,\n                 gpu_mem_peak_mib=[max(row[i][1] for row in gmem if len(row) > i) for i in range(max(map(len, gmem)))] if gmem else None,\n                 preemptions=m1.get(PREEMPT, 0) - m0.get(PREEMPT, 0), t0=t0, t1=t1)\n        if any(x >= 3 for x in stalls):\n            r["status"] = "INCONCLUSIVE_STALLED_STREAM"\n        elif sum(g.is_alive() for g in ss) < min(level, best):\n            r["status"] = "INCONCLUSIVE_STREAMS_ENDED_IN_WINDOW"\n        elif any(x.get(PROMPT) != m0.get(PROMPT) for x in mids + [m1]):\n            r["status"] = "INCONCLUSIVE_PREFILL_IN_WINDOW"\n        elif r["preemptions"] > 0:\n            r["status"] = "INCONCLUSIVE_PREEMPTION"\n        elif gen <= 0:\n            r["status"] = "FAIL_NO_DECODE"\n        elif min(runs or [0]) < level and (len(set(runs)) > 1 or len(set(waits)) > 1):\n            r["status"] = "INCONCLUSIVE_RUNNING_SET_CHANGED"      # KV-bound window must hold one fixed set\n        elif min(runs or [0]) < 0.8 * max(runs or [1]):\n            r["status"] = "INCONCLUSIVE_RUNNING_UNSTABLE"\n        else:\n            r["status"] = ("STEADY" if min(runs or [0]) == max(runs or [0]) == level else\n                           "CAPACITY_LIMITED" if max(runs or [0]) < level else "INCONCLUSIVE_RUNNING_ABOVE_LEVEL")\n        return r\n    finally:\n        errs = sorted({g.err for g in ss if g.err})[:3]           # rev 3: snapshot before stop() sets _halt\n        for g in ss:\n            g.stop()\n        for g in ss:\n            g.join(10)\n        r["client_leak"] = any(g.is_alive() for g in ss)\n        r["stream_errors"] = errs\n        gb = [degenerate("".join(g.text)[-4000:]) if g.tokens else None for g in ss]\n        r["garbage"] = {"streams": len(ss), "with_output": sum(1 for g in ss if g.tokens),\n                        "garbage": [x for x in gb if x], "sample_tail": "".join(ss[0].text)[-200:] if ss and ss[0].tokens else ""}\n        if r["garbage"]["garbage"] and r.get("status") in ("STEADY", "CAPACITY_LIMITED"):\n            r["status"] = "FAIL_GARBAGE_UNDER_LOAD"\n        r["idle_after"] = wait_idle(port)\n\n\ndef steady_multi(jobs, secs=60.0, **kw):\n    """Several servers at once (TP1 replicas): jobs = [(port, prompts, level)]; one shared window (barrier). The aggregate\n    counts only when every window is STEADY / CAPACITY_LIMITED and the windows overlap >= 80 % of the shortest."""\n    out = [None] * len(jobs)\n    bar = threading.Barrier(len(jobs))\n\n    def one(i, j):\n        try:\n            out[i] = steady(j[0], j[1], j[2], secs=secs, barrier=bar, **kw)\n        except Exception as e:          # rev 3: a dead replica is recorded, not printed to stderr only\n            out[i] = {"status": "ERROR_REPLICA", "error": f"{type(e).__name__}: {e}"[:300], "port": j[0]}\n            try:\n                bar.abort()\n            except Exception:\n                pass\n    th = [threading.Thread(target=one, args=(i, j), daemon=True) for i, j in enumerate(jobs)]\n    for t in th:\n        t.start()\n    for t in th:\n        t.join()\n    good = all(o and o.get("status") in ("STEADY", "CAPACITY_LIMITED") for o in out)\n    ov = (min(o["t1"] for o in out) - max(o["t0"] for o in out)) if good else 0\n    aligned = good and ov > 0.8 * min(o["window_sec"] for o in out)\n    return {"replicas": out, "aligned": aligned, "overlap_sec": round(ov, 1),\n            "tok_s": round(sum(o["tok_s"] for o in out), 1) if aligned else None,\n            "running_mean": round(sum(o["running_mean"] for o in out), 2) if aligned else None,\n            "status": "AGGREGATE_STEADY" if aligned else "INCONCLUSIVE_REPLICA_WINDOWS"}\n\n\ndef wait_idle(port, wait=90.0):\n    end, hits = time.perf_counter() + wait, 0\n    while time.perf_counter() < end:\n        try:\n            m = prom(port)\n            hits = hits + 1 if m.get(RUN, 0) == 0 and m.get(WAIT, 0) == 0 else 0\n        except Exception:\n            hits = 0\n        if hits >= 3:\n            return True\n        time.sleep(2)\n    return False\n\n\n# ------------------------------------------------------------------ profiler (FULL-graph proof + per-category time)\n# rev 3: matched on the kernel BASE name, attention before GDN ("fla" is a substring of "flashinfer"), real 0.32 GDN names\nCATS = [("nccl", ("nccl", "allreduce", "all_reduce", "allgather", "all_gather")),\n        ("attention", ("flashinfer", "flash_fwd", "unified_attention", "paged", "attn", "attention", "batchdecode", "batchprefill")),\n        ("gdn_linear_attn", ("gdn", "delta_rule", "chunk_", "recurrent", "causal_conv", "l2norm", "solve_tril", "merge_16x16",\n                             "recompute_w_u", "kkt", "cumsum", "post_conv", "layer_norm_gated", "gated_delta")),\n        ("gemm_marlin", ("marlin",)),\n        ("gemm_other", ("gemm", "cutlass", "xmma", "cublas", "sm89_", "ampere_", "sm80_")),\n        ("sampling", ("sampl", "topk", "top_k", "argmax", "softmax", "rejection"))]\n\n\ndef _cat(name):\n    base = str(name).split("<")[0].split("(")[0].lower()\n    return next((k for k, pats in CATS if any(p in base for p in pats)), "other")\n\n\ndef trace_summary(pdir):\n    files = sorted(Path(pdir).rglob("*.pt.trace.json*"), key=lambda p: p.stat().st_mtime)\n    if not files:\n        return {"error": "NO_TRACE_FILE", "listing": [str(p) for p in Path(pdir).rglob("*")][:20]}\n    f = next((p for p in files if re.search(r"rank0\\.|rank[-_]?0[._]|tp0_", p.name)), files[0])\n    with (gzip.open if f.name.endswith(".gz") else open)(f, "rt", encoding="utf-8") as fh:\n        tr = json.load(fh)\n    ev = tr.get("traceEvents", [])\n    kern = [e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")).lower() in ("kernel", "gpu_kernel")]\n    rt = [e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")).lower() in ("cuda_runtime", "cuda_driver")]\n    is_graph = lambda e: "GraphLaunch" in str(e.get("name", ""))\n    is_kl = lambda e: re.search(r"LaunchKernel|cuLaunchKernel", str(e.get("name", ""))) is not None\n    # per-forward counts inside gpu_model_runner forward annotations (needs VLLM_CUSTOM_SCOPES_FOR_PROFILING=1)\n    fw = [e for e in ev if e.get("ph") == "X" and "forward" in str(e.get("name", "")).lower()\n          and "gpu_model_runner" in str(e.get("name", "")).lower()]\n    per_fw = []\n    for a in fw[:400]:\n        lo, hi, tid = a.get("ts", 0), a.get("ts", 0) + a.get("dur", 0), a.get("tid")\n        inside = [e for e in rt if e.get("tid") == tid and lo <= e.get("ts", -1) <= hi]\n        per_fw.append([sum(1 for e in inside if is_graph(e)), sum(1 for e in inside if is_kl(e))])\n    corr = {(e.get("args") or {}).get("correlation"): str(e.get("name", "")) for e in rt\n            if (e.get("args") or {}).get("correlation") is not None}\n    per, tot, names = {k: 0.0 for k, _ in CATS + [("other", ())]}, 0.0, {}\n    src = {k: {"graph": 0, "eager": 0, "unknown": 0} for k in per}\n    for e in kern:\n        d = float(e.get("dur", 0.0))\n        tot += d\n        names[e.get("name", "")] = names.get(e.get("name", ""), 0.0) + d\n        k = _cat(e.get("name", ""))\n        per[k] += d\n        launcher = corr.get((e.get("args") or {}).get("correlation"))\n        src[k]["graph" if launcher and "GraphLaunch" in launcher else ("eager" if launcher else "unknown")] += 1\n    tot_k = sum(v["graph"] + v["eager"] for v in src.values())\n    eager_share = round(sum(v["eager"] for v in src.values()) / tot_k, 3) if tot_k else None\n    fw_g = statistics.median([x[0] for x in per_fw]) if per_fw else None\n    fw_k = statistics.median([x[1] for x in per_fw]) if per_fw else None\n    if per_fw:\n        verdict = "FULL" if fw_g == 1 and fw_k == 0 else ("PIECEWISE" if fw_g and fw_g > 1 else ("EAGER" if not fw_g else "MIXED"))\n        basis = "per_forward_annotation"\n    elif tot_k:\n        att = src["attention"]\n        verdict = ("FULL" if att["eager"] == 0 and (eager_share or 0) < 0.05 else\n                   "PIECEWISE" if att["eager"] > 0 and any(v["graph"] for v in src.values()) else\n                   "EAGER" if not any(v["graph"] for v in src.values()) else "MIXED")\n        basis = "cupti_correlation"\n    else:\n        verdict, basis = "UNKNOWN", "none"\n    return {"file": f.name, "files": len(files), "kernels": len(kern), "kernel_us_sum": round(tot),\n            "graph_launches": sum(1 for e in rt if is_graph(e)), "kernel_launches": sum(1 for e in rt if is_kl(e)),\n            "forwards": len(per_fw), "per_forward_median_graph_launches": fw_g, "per_forward_median_kernel_launches": fw_k,\n            "launch_source": src, "eager_kernel_share": eager_share, "graph_verdict": verdict, "verdict_basis": basis,\n            "by_category_pct": {k: round(100 * v / tot, 1) for k, v in per.items()} if tot else {},\n            "top_kernels_us": dict(sorted(((k, round(v)) for k, v in names.items()), key=lambda x: -x[1])[:25]),\n            "note": "kernel time sums overlapping streams; NCCL time includes rank-skew waiting"}\n\n\ndef profile(port, pdir, secs=6.0, delay_iters=20, max_iters=40):\n    """POST /start_profile (bounded by iterations; falls back to a bare POST), decode, POST /stop_profile, summary."""\n    rec = {}\n    try:\n        U.urlopen(U.Request(f"http://127.0.0.1:{port}/start_profile", method="POST",\n                            data=json.dumps({"delay_iterations": delay_iters, "max_iterations": max_iters}).encode(),\n                            headers={"Content-Type": "application/json"}), timeout=900).read()\n        rec["start_profile"] = "ok_bounded"\n    except Exception as e:\n        rec["start_profile_bounded_error"] = repr(e)[:200]\n        try:\n            U.urlopen(U.Request(f"http://127.0.0.1:{port}/start_profile", data=b"", method="POST"), timeout=900).read()\n            rec["start_profile"] = "ok"\n        except Exception as e2:\n            rec["start_profile"] = repr(e2)[:200]\n    time.sleep(secs)\n    try:\n        U.urlopen(U.Request(f"http://127.0.0.1:{port}/stop_profile", data=b"", method="POST"), timeout=900).read()\n        rec["stop_profile"] = "ok"\n    except Exception as e:\n        rec["stop_profile"] = repr(e)[:200]\n    time.sleep(20)\n    try:\n        rec["trace"] = trace_summary(pdir)\n    except Exception as e:\n        rec["trace"] = {"error": f"TRACE_PARSE {type(e).__name__}: {e}"[:300]}\n    return rec\n\n\ndef profiled_load(port, prompts, level, pdir, secs=6.0, max_new=8192, model="q27", prefill_tok_s=700.0):\n    """`level` decoding streams (own window: the profiler slows decode), profile() only after prefill settled exactly\n    as in steady() (prompt counter + preemptions flat, every running request emitted and gaining tokens)."""\n    plen = len(prompts[0]) if prompts else 0\n    settle = max(300.0, 120.0 + level * plen / prefill_tok_s)\n    ss = [Stream(port, prompts[i % len(prompts)], max_new, 1.0, 4321 + i, model=model) for i in range(level)]\n    for g in ss:\n        g.start()\n    rec, t0, flat, ok, prev, last_tok = {"level": level}, time.perf_counter(), None, 0, None, [0] * level\n    try:\n        while time.perf_counter() - t0 < settle:\n            time.sleep(2)\n            try:\n                m = prom(port)\n            except Exception:\n                continue\n            run, wait, pt, pre = int(m.get(RUN, 0)), int(m.get(WAIT, 0)), m.get(PROMPT), m.get(PREEMPT, 0)\n            now_tok = [len(g.tokens) for g in ss]\n            gaining = sum(1 for a, b in zip(last_tok, now_tok) if b > a)\n            last_tok = now_tok\n            first = sum(g.t_first is not None for g in ss)\n            calm = run > 0 and prev is not None and (pt, pre) == prev and first >= run and gaining >= run\n            if calm and run >= level and wait == 0:\n                ok += 1\n            elif calm and wait > 0:\n                flat = flat or time.perf_counter()\n                ok = ok + 1 if time.perf_counter() - flat > 30 else 0\n            else:\n                ok, flat = 0, None\n            prev = (pt, pre)\n            if ok >= 3:\n                break\n        rec["settled"] = ok >= 3\n        try:\n            rec["running_at_start"] = prom(port).get(RUN)\n        except Exception:\n            pass\n        rec.update(profile(port, pdir, secs))\n        if not rec["settled"] and isinstance(rec.get("trace"), dict):\n            rec["trace"]["note2"] = "prefill had not settled: the trace may mix prefill and decode"\n        return rec\n    finally:\n        for g in ss:\n            g.stop()\n        for g in ss:\n            g.join(10)\n        rec["idle_after"] = wait_idle(port)\n'
    (TMP / "e23_lab.py").write_text(LAB_SRC)
    sys.path.insert(0, str(TMP))
    import e23_lab as L
    P60532 = 'diff --git a/tests/v1/attention/test_attention_backends.py b/tests/v1/attention/test_attention_backends.py\nindex bb8e154b2f37..0f911594931b 100644\n--- a/tests/v1/attention/test_attention_backends.py\n+++ b/tests/v1/attention/test_attention_backends.py\n@@ -500,8 +500,8 @@ def _test_backend_correctness(\n     if max_num_batched_tokens is not None:\n         vllm_config.scheduler_config.max_num_batched_tokens = max_num_batched_tokens\n     if num_speculative_tokens > 0:\n-        vllm_config.speculative_config = SimpleNamespace(\n-            num_speculative_tokens=num_speculative_tokens\n+        vllm_config.speculative_config = SpeculativeConfig(\n+            method="ngram", num_speculative_tokens=num_speculative_tokens\n         )\n     vllm_config.cache_config.cache_dtype = kv_cache_dtype\n     device = torch.device(f"{DEVICE_TYPE}:0")\n@@ -1205,6 +1205,7 @@ def test_flashinfer_varlen_cudagraph_capability(\n         use_v2_model_runner=True,\n     )\n     monkeypatch.setattr(fi, "can_use_trtllm_attention", lambda *_, **__: True)\n+    monkeypatch.setattr(fi.current_platform, "has_device_capability", lambda _: True)\n     monkeypatch.setattr(\n         fi.FlashInferMetadataBuilder,\n         "_get_flashinfer_trtllm_api_decode_kernel",\n@@ -1715,16 +1716,17 @@ def causal_mask_mod(\n \n @pytest.mark.skipif(\n     AttentionBackendEnum.FLASHINFER not in BACKENDS_TO_TEST\n-    or not current_platform.is_device_capability_family(120),\n-    reason="NVFP4 XQA decode requires SM12x.",\n+    or not any(current_platform.is_device_capability_family(f) for f in (80, 120)),\n+    reason="FlashInfer decodes an NVFP4 cache with XQA on SM12x and fa2 on SM8x.",\n )\n @pytest.mark.parametrize("use_cuda_graph", [False, True])\n @pytest.mark.parametrize("q_len", [1, 4])\n-def test_flashinfer_xqa_nvfp4_decode_correctness(\n+def test_flashinfer_nvfp4_decode_correctness(\n     default_vllm_config, q_len, use_cuda_graph\n ):\n     """Single-token and spec-decode queries over an NVFP4 cache decode with\n-    XQA, eager and replayed, and match the fa2 prefill that wrote the cache."""\n+    XQA on SM12x and fa2 on SM8x, eager and replayed, and match the fa2\n+    prefill that wrote the cache."""\n     import unittest.mock\n \n     from vllm.v1.attention.backends import flashinfer as fi\n@@ -1801,15 +1803,46 @@ def run(query_lens, use_cuda_graph=False):\n     # The full prefill writes every token through the NVFP4 writer; its fa2\n     # output for the last q_len tokens of each request is the reference.\n     _, reference = run(seq_lens)\n-    xqa_fn = fi.flashinfer_xqa_batch_decode_with_kv_cache\n-    with unittest.mock.patch.object(\n-        fi, "flashinfer_xqa_batch_decode_with_kv_cache", wraps=xqa_fn\n-    ) as xqa:\n+    xqa = current_platform.is_device_capability_family(120)\n+    name = "flashinfer_xqa_batch_decode_with_kv_cache" if xqa else "fast_plan_decode"\n+    with unittest.mock.patch.object(fi, name, wraps=getattr(fi, name)) as decode:\n         rows, output = run([q_len] * len(seq_lens), use_cuda_graph)\n-    assert xqa.called\n+    assert decode.called\n+    assert xqa or decode.call_args.kwargs["q_len_per_req"] == q_len\n     torch.testing.assert_close(output, reference[rows], atol=2e-2, rtol=2e-2)\n \n \n+@pytest.mark.skipif(\n+    AttentionBackendEnum.FLASHINFER not in BACKENDS_TO_TEST\n+    or current_platform.has_device_capability(90),\n+    reason="FlashInfer fa2 spec decode runs before SM90.",\n+)\n+@pytest.mark.parametrize("use_cuda_graph", [False, True])\n+def test_flashinfer_fa2_spec_decode_correctness(default_vllm_config, use_cuda_graph):\n+    """Uniform spec-decode queries take fa2 decode and match SDPA."""\n+    import unittest.mock\n+\n+    from vllm.v1.attention.backends import flashinfer as flashinfer_backend\n+\n+    def causal_mask_mod(b, h, q_idx, kv_idx, *, context_len):\n+        return (q_idx + context_len) >= kv_idx\n+\n+    with unittest.mock.patch.object(\n+        flashinfer_backend,\n+        "fast_plan_decode",\n+        wraps=flashinfer_backend.fast_plan_decode,\n+    ) as plan:\n+        _test_backend_correctness(\n+            BatchSpec(seq_lens=[32, 40, 1024], query_lens=[4, 4, 4]),\n+            "meta-llama/Meta-Llama-3-8B",\n+            [AttentionBackendEnum.FLASHINFER],\n+            causal_mask_mod,\n+            use_cuda_graph=use_cuda_graph,\n+            num_speculative_tokens=3,\n+        )\n+    assert plan.call_args.kwargs["q_len_per_req"] == 4\n+\n+\n if current_platform.is_rocm():\n     # FLASH_ATTN is not supported on ROCm\n     SLIDING_WINDOW_BACKENDS_TO_TEST = [\ndiff --git a/vllm/v1/attention/backends/flashinfer.py b/vllm/v1/attention/backends/flashinfer.py\nindex bb9e1b0e4577..1183259d884e 100755\n--- a/vllm/v1/attention/backends/flashinfer.py\n+++ b/vllm/v1/attention/backends/flashinfer.py\n@@ -111,6 +111,11 @@ def _nvfp4_kv_on_fa2() -> bool:\n     return any(current_platform.is_device_capability_family(f) for f in (80, 90, 120))\n \n \n+def _use_fa2_native_decode() -> bool:\n+    # Below SM90, native decode runs fa2, which takes uniform multi-token queries.\n+    return not current_platform.has_device_capability(90)\n+\n+\n _KVPair = tuple[torch.Tensor, torch.Tensor]\n \n \n@@ -885,9 +890,9 @@ def __init__(\n         )\n         if self.enable_cuda_graph:\n             # For full cudagraph capture, one `decode_wrapper` for each batch\n-            # size is needed for FlashInfer.\n+            # size and query length is needed for FlashInfer.\n             self._decode_wrappers_cudagraph: dict[\n-                int, BatchDecodeWithPagedKVCacheWrapper\n+                tuple[int, int], BatchDecodeWithPagedKVCacheWrapper\n             ] = {}\n             self._decode_cudagraph_max_bs = (1 + num_spec_tokens) * max_num_reqs\n             if self.compilation_config.max_cudagraph_capture_size is not None:\n@@ -1021,10 +1026,12 @@ def __init__(\n             == FlashInferDecodeKernel.TRTLLM_GEN\n             and not self.use_dcp\n         )\n+        self.use_fa2_native_decode = _use_fa2_native_decode()\n         self._init_reorder_batch_threshold(\n             1,\n             supports_spec_as_decode=(\n                 self.flashinfer_trtllm_api_decode_kernel is not None\n+                or self.use_fa2_native_decode\n             ),\n             # trtllm-gen decode receives no cp_rank/global-seq-len information,\n             # so its end-aligned causal mask is wrong for q_len > 1 over the\n@@ -1153,6 +1160,12 @@ def get_cudagraph_support(\n             return AttentionCGSupport.UNIFORM_SINGLE_TOKEN_DECODE\n \n         kv_specs = iter_layer_specs(kv_cache_spec)\n+        if _use_fa2_native_decode():\n+            # fa2 decodes uniform causal spec queries.\n+            if kv_specs and not vllm_config.attention_config.use_non_causal:\n+                return AttentionCGSupport.UNIFORM_BATCH\n+            return AttentionCGSupport.UNIFORM_SINGLE_TOKEN_DECODE\n+\n         num_qo_heads = vllm_config.model_config.get_num_attention_heads(\n             vllm_config.parallel_config\n         )\n@@ -1197,8 +1210,10 @@ def get_varlen_cudagraph_max_query_len(\n     ) -> int | None:\n         # The uniform gate already covers DCP, head geometry, the XQA head-dim\n         # limit and causality, and yields None for subclasses that force NEVER.\n+        # The native fa2 decode wrapper takes only uniform query lengths.\n         if (\n-            cls.get_cudagraph_support(vllm_config, kv_cache_spec)\n+            _use_fa2_native_decode()\n+            or cls.get_cudagraph_support(vllm_config, kv_cache_spec)\n             != AttentionCGSupport.UNIFORM_BATCH\n         ):\n             return None\n@@ -1417,9 +1432,13 @@ def _get_prefill_wrapper(\n         assert self._prefill_wrapper is not None\n         return self._prefill_wrapper\n \n-    def _get_decode_wrapper(self, batch_size: int, use_cudagraph: bool = False):\n+    def _get_decode_wrapper(\n+        self, batch_size: int, use_cudagraph: bool = False, q_len_per_req: int = 1\n+    ):\n         if use_cudagraph:\n-            decode_wrapper = self._decode_wrappers_cudagraph.get(batch_size, None)\n+            decode_wrapper = self._decode_wrappers_cudagraph.get(\n+                (batch_size, q_len_per_req), None\n+            )\n         else:\n             decode_wrapper = self._decode_wrapper\n \n@@ -1434,7 +1453,13 @@ def _get_decode_wrapper(self, batch_size: int, use_cudagraph: bool = False):\n                 paged_kv_last_page_len = None\n             # NVFP4 KV cache requires the trtllm-gen backend inside\n             # the wrapper on SM100; SM8x, SM90 and SM12x read it with fa2.\n-            backend = "trtllm-gen" if self.nvfp4_trtllm else "auto"\n+            backend = (\n+                "trtllm-gen"\n+                if self.nvfp4_trtllm\n+                else "fa2"\n+                if self.use_fa2_native_decode\n+                else "auto"\n+            )\n             decode_wrapper = BatchDecodeWithPagedKVCacheWrapper(\n                 self._get_workspace_buffer(),\n                 get_flashinfer_layout_string(self.kv_cache_layout),\n@@ -1451,7 +1476,9 @@ def _get_decode_wrapper(self, batch_size: int, use_cudagraph: bool = False):\n \n             # save the decode wrapper\n             if use_cudagraph:\n-                self._decode_wrappers_cudagraph[batch_size] = decode_wrapper\n+                self._decode_wrappers_cudagraph[batch_size, q_len_per_req] = (\n+                    decode_wrapper\n+                )\n             else:\n                 self._decode_wrapper = decode_wrapper\n \n@@ -1890,10 +1917,14 @@ def build(\n                     and pure_decode\n                     and num_decode_tokens <= self._decode_cudagraph_max_bs\n                 )\n-                num_input_tokens = num_decode_tokens\n+                num_input_tokens, q_len_per_req = num_decode_tokens, 1\n+                if self.use_fa2_native_decode:\n+                    # fa2 plans uniform spec decodes per request.\n+                    num_input_tokens = num_decodes\n+                    q_len_per_req = num_decode_tokens // num_decodes\n \n                 decode_wrapper = self._get_decode_wrapper(\n-                    num_input_tokens, use_cudagraph\n+                    num_input_tokens, use_cudagraph, q_len_per_req\n                 )\n                 # Use the persistent buffer with padding length,\n                 # instead of the same address but chunked version\n@@ -1942,6 +1973,7 @@ def build(\n                     o_data_type=o_dtype,\n                     fixed_split_size=self.decode_fixed_split_size,\n                     disable_split_kv=self.disable_split_kv,\n+                    q_len_per_req=q_len_per_req,\n                 )\n                 attn_metadata.decode = FIDecode(wrapper=decode_wrapper)\n         return attn_metadata\n@@ -2807,6 +2839,7 @@ def fast_plan_decode(\n     non_blocking: bool = True,\n     fixed_split_size: int = -1,\n     disable_split_kv: bool = False,\n+    q_len_per_req: int = 1,\n ) -> None:\n     """A faster version of BatchDecodeWithPagedKVCacheWrapper::plan used for\n     cudagraph capture/replay, while the no cudagraph version turns back\n@@ -2851,6 +2884,7 @@ def fast_plan_decode(\n             seq_lens=seq_lens_cpu,\n             fixed_split_size=fixed_split_size,\n             disable_split_kv=disable_split_kv,\n+            q_len_per_req=q_len_per_req,\n         )\n         self.vllm_first_call = False\n         return\n@@ -2878,6 +2912,7 @@ def fast_plan_decode(\n         non_blocking=non_blocking,\n         fixed_split_size=fixed_split_size,\n         disable_split_kv=disable_split_kv,\n+        q_len_per_req=q_len_per_req,\n     )\n \n \n'
    M = INP / "models" / "foysalemonshanto"
    FP8 = find("config.json", M / "qwen3-8-27b-fp8-repacked-v1").parent
    _i4 = find("config.json", M / "qwen3-8-27b-int4-redhat", must=False)
    INT4 = _i4.parent if _i4 else None
    _mm = find("q27_e23_p1_manifest.json", M / "qwen3-8-27b-w4a16-merlin", must=False)
    MERLIN = _mm.parent if _mm else None
    MERLIN_PAY = find("merlin_offline_tar.bin", M / "e23-merlin-env", must=False)
    EAGLE_PAY = find("eagle_offline_tar.bin", M / "e23-eagle4-env", must=False)
    _ed = find("config.json", M / "qwen3-8-27b-eagle4", must=False)
    EAGLE_DRAFT = _ed.parent if _ed else None
    KITS = find("q27_e23_kernels_manifest.json", INP).parent
    WHEELHOUSE = find("install_wheelhouse.py", INP / "datasets" / "foysalemonshanto" / "vllm-0-32-0-cu13-wheelhouse")
    ARC = find("arc-agi_evaluation_challenges.json")
    R["inputs"] = {k: str(v) for k, v in dict(FP8=FP8, INT4=INT4, MERLIN=MERLIN, MERLIN_PAY=MERLIN_PAY, EAGLE_PAY=EAGLE_PAY,
                                              EAGLE_DRAFT=EAGLE_DRAFT, KITS=KITS, WHEELHOUSE=WHEELHOUSE, ARC=ARC).items()}
    MERLIN_OK = False
    if MERLIN:
        mt = json.loads((MERLIN / "q27_e23_p1_manifest.json").read_text())
        MERLIN_OK = all((MERLIN / n).is_file() and (MERLIN / n).stat().st_size == s
                        for n, (s, _) in mt["files"].items() if n != ".gitattributes")
    R["inputs"]["merlin_files_ok"] = MERLIN_OK
    km = json.loads((KITS / "q27_e23_kernels_manifest.json").read_text())       # rev 3: the kit files are what we built
    R["inputs"]["kits_sha_ok"] = {n: (KITS / n).is_file() and hashlib.sha256((KITS / n).read_bytes()).hexdigest() == v.get("sha256")
                                  for n, v in km.items() if isinstance(v, dict) and n.endswith((".so", ".py"))}
    R["identity"] = {k: L.identity(v) for k, v in dict(fp8=FP8, int4=INT4, merlin=MERLIN, eagle_draft=EAGLE_DRAFT).items() if v}
    ABORT = {"why": None}
    L.start_watchdog(HARD_STOP, lambda: ABORT.update(why="HARD_STOP reached: watchdog killed live servers"))
    log("MODE", MODE, "PART", PART, "| budget", BUDGET // 60, "min |", json.dumps(R["inputs"])[:900])
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 2 — engines (stock vLLM 0.32, Merlin, EAGLE fork)
    def run(name, cmd, timeout=3600, env=None):
        t = time.time()
        try:
            with open(LOGS / f"{name}.log", "w") as f:
                r = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], stdout=f, stderr=subprocess.STDOUT, timeout=timeout,
                                   env={**os.environ, **(env or {})})
            rc = r.returncode
        except subprocess.TimeoutExpired:
            rc = -9
        txt = (LOGS / f"{name}.log").read_text(errors="replace")
        R["setup"][name] = {"rc": rc, "sec": round(time.time() - t, 1), "tail": txt[-1500:]}
        log(name, "rc", rc, f"{time.time() - t:.0f}s")
        save()
        return rc, txt


    SHIM = TMP / "e23_libcuda"          # FlashInfer JIT links -lcuda; Kaggle has only libcuda.so.1
    SHIM.mkdir(exist_ok=True)
    for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/compat", "/usr/lib64"):
        c = sorted(Path(d).glob("libcuda.so.1")) if Path(d).is_dir() else []
        if c and not (SHIM / "libcuda.so").exists():
            (SHIM / "libcuda.so").symlink_to(c[0])
    if FAKE:
        STOCK = {"vllm": [sys.executable, FAKE], "env": dict(os.environ), "py": sys.executable, "name": "stock"}
    else:
        rc, txt = run("install_stock", f"{sys.executable} {WHEELHOUSE} {TMP}/vllm-venv", timeout=2400)
        kv = dict(l.split("=", 1) for l in txt.splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))
        if rc or "CUDA_HOME" not in kv:                                   # rev 3: a clear failure, not a KeyError
            raise RuntimeError("stock vLLM 0.32 install failed (logs/install_stock.log)")
        env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH", "LD_PRELOAD")}
        env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"{TMP}/vllm-venv/bin:{kv['CUDA_HOME']}/bin:{env.get('PATH', '')}",
                   LIBRARY_PATH=f"{SHIM}:{env.get('LIBRARY_PATH', '')}", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   VLLM_NO_USAGE_STATS="1", VLLM_CACHE_ROOT=f"{TMP}/vllm-cache", VLLM_ALLOW_LONG_MAX_MODEL_LEN="1")
        STOCK = {"vllm": [f"{TMP}/vllm-venv/bin/vllm", "serve"], "env": env, "py": kv["VENV_PYTHON"], "cuda_home": kv["CUDA_HOME"],
                 "name": "stock"}
        R["setup"]["stock_version"] = subprocess.run([kv["VENV_PYTHON"], "-c", "import vllm; print(vllm.__version__)"],
                                                     capture_output=True, text=True, env=env).stdout.strip()
    HELP = {}


    def prefetch_bg(paths, tag):
        """rev 3: warm the page cache of a cold 17-30 GB /kaggle/input checkpoint in the background (many readers)."""
        def go():
            R["setup"]["prefetch_" + tag] = L.prefetch([p for p in paths if p]) if not FAKE else {"fake": True}
            save()
        th = threading.Thread(target=go, daemon=True)
        th.start()
        return th


    MERLIN_ENG = None


    def merlin_install():
        """P2 payload (.bin = a tar Kaggle does not unpack) -> install_merlin_offline.sh (UV_OFFLINE); payload deleted after;
        a failure is cached so later Merlin steps fail fast (rev 3)."""
        global MERLIN_ENG
        if MERLIN_ENG is not None:
            if MERLIN_ENG.get("failed"):
                raise RuntimeError("Merlin install failed earlier: " + MERLIN_ENG["failed"])
            return MERLIN_ENG
        if not MERLIN_OK or MERLIN_PAY is None:
            MERLIN_ENG = {"failed": "Merlin checkpoint / env not attached or incomplete"}
            raise RuntimeError(MERLIN_ENG["failed"])
        if FAKE:
            MERLIN_ENG = {"fake": True}
            return MERLIN_ENG
        try:
            pay_root = TMP / "merlin_pay"
            shutil.rmtree(pay_root, ignore_errors=True)
            pay_root.mkdir()
            t = time.time()
            with tarfile.open(MERLIN_PAY) as tf:
                tf.extractall(pay_root, filter="data")
            R["setup"]["merlin_extract_sec"] = round(time.time() - t, 1)
            pay = pay_root / "merlin_offline"
            shutil.rmtree(TMP / "merlin_off", ignore_errors=True)
            rc, txt = run("install_merlin", f"bash {pay}/install_merlin_offline.sh {pay} {TMP}/merlin_off", timeout=5400)
            kv = dict(re.findall(r"^(MERLIN_[A-Z]+)=(.*)$", txt, re.M))
            shutil.rmtree(pay_root, ignore_errors=True)
            if rc or "INSTALL OK" not in txt:
                raise RuntimeError("Merlin offline install failed (logs/install_merlin.log)")
            sys.path.insert(0, kv["MERLIN_TREE"] + "/tools/merlin")
            import launch as ML
            site = subprocess.run([kv["MERLIN_VENV"] + "/bin/python", "-c", "import sysconfig;print(sysconfig.get_paths()['purelib'])"],
                                  capture_output=True, text=True).stdout.strip()
            MERLIN_ENG = {"kv": kv, "ML": ML, "cuda_home": site + "/nvidia/cu13"}
            return MERLIN_ENG
        except Exception as e:
            MERLIN_ENG = {"failed": f"{type(e).__name__}: {e}"[:300]}
            raise


    def merlin_server(name, profile, gpus, port, kv_bytes, seqs=None, capture=None):
        """Merlin's own launch.build() (pure: the profile's flags + env, no host refusal), our max len / port last, and
        seqs / capture sizes only when given (None = the profile's own, the TP1 memory fallback)."""
        eng = merlin_install()
        extra = ["--served-model-name", "q27", "--max-model-len", str(MAX_LEN), "--port", str(port)]
        if seqs:
            extra += ["--max-num-seqs", str(seqs)]
        if capture:
            extra += ["--compilation-config", json.dumps({"cudagraph_mode": "FULL_AND_PIECEWISE", "cudagraph_capture_sizes": capture,
                                                          "inductor_compile_config": {"test_configs.force_filter_reduction_configs": True,
                                                                                      "dynamic_scale_rblock": False}})]
        if FAKE:
            return L.Server(name, [sys.executable, FAKE, str(MERLIN), "--tensor-parallel-size", str(len(gpus.split(","))),
                                   "--max-num-seqs", str(seqs or 2), "--port", str(port)], os.environ, port, LOGS, gpus=gpus)
        kvv = eng["kv"]
        base = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH", "CUDA_HOME", "LD_PRELOAD", "CC", "CXX")}
        base.update(GPUS=gpus, PORT=str(port), KV_BYTES=str(kv_bytes), P2P="off", VENV=kvv["MERLIN_VENV"])
        plan = eng["ML"].build("27b", profile, str(MERLIN), base, extra=extra, checkpoint="27b", status="untested")
        env = dict(plan.env)
        # rev 3: -lcuda shim (FlashInfer JIT), system GCC at serve time (GCC 13 was only the install rule), per-port caches
        env.update(LD_PRELOAD=kvv["MERLIN_SHIM"], CUDA_HOME=eng["cuda_home"], VLLM_USE_RUST_FRONTEND="0", HF_HUB_OFFLINE="1",
                   TRANSFORMERS_OFFLINE="1", LIBRARY_PATH=f"{SHIM}:{os.environ.get('LIBRARY_PATH', '')}",
                   PATH=f"{kvv['MERLIN_VENV']}/bin:{eng['cuda_home']}/bin:{os.environ.get('PATH', '')}",
                   VLLM_CACHE_ROOT=f"{TMP}/merlin-cache", TRITON_CACHE_DIR=f"{TMP}/merlin-triton-{port}",
                   FLASHINFER_WORKSPACE_BASE=f"{TMP}/merlin-fi")
        for k in ("CC", "CXX", "CUDAHOSTCXX"):
            env.pop(k, None)
        dp = getattr(eng["ML"], "distributed_ports", None)                   # Merlin's own --instances port scheme
        if dp:
            try:
                ports = dp(port)
                env.update({k: str(v) for k, v in (ports.items() if isinstance(ports, dict) else
                                                   zip(("MASTER_PORT", "VLLM_PORT"), ports))})
            except Exception:
                env.update(MASTER_PORT=str(29500 + 10 * (port - 8100)), VLLM_PORT=str(29505 + 10 * (port - 8100)))
        else:
            env.update(MASTER_PORT=str(29500 + 10 * (port - 8100)), VLLM_PORT=str(29505 + 10 * (port - 8100)))
        argv = plan.argv()
        HELP[name] = L.help_check("merlin", argv[:argv.index("serve") + 1], env, argv)
        return L.Server(name, argv, env, port, LOGS, gpus=gpus)


    EAGLE_ENG = None


    def eagle_install():
        global EAGLE_ENG
        if EAGLE_ENG is not None:
            if EAGLE_ENG.get("failed"):
                raise RuntimeError("EAGLE install failed earlier: " + EAGLE_ENG["failed"])
            return EAGLE_ENG
        if FAKE or EAGLE_PAY is None or EAGLE_DRAFT is None:
            return None
        try:
            pay_root = TMP / "eagle_pay"
            shutil.rmtree(pay_root, ignore_errors=True)
            pay_root.mkdir()
            with tarfile.open(EAGLE_PAY) as tf:
                tf.extractall(pay_root, filter="data")
            pay = pay_root / "eagle_offline"
            rc, txt = run("install_eagle", f"bash {pay}/install_eagle_offline.sh {pay} {TMP}/eagle-off", timeout=2400)
            shutil.rmtree(pay_root, ignore_errors=True)
            if rc or "INSTALL OK" not in txt:
                raise RuntimeError("EAGLE offline install failed (logs/install_eagle.log)")
            env = dict(STOCK["env"])                                           # rev 3: the fork's own PATH / caches
            env.update(PATH=f"{TMP}/eagle-off/bin:{env.get('PATH', '')}", VLLM_CACHE_ROOT=f"{TMP}/eagle-cache",
                       VLLM_USE_V2_MODEL_RUNNER="1", TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="4")
            EAGLE_ENG = {"vllm": [f"{TMP}/eagle-off/bin/vllm", "serve"], "env": env, "name": "eagle"}
            return EAGLE_ENG
        except Exception as e:
            EAGLE_ENG = {"failed": f"{type(e).__name__}: {e}"[:300]}
            raise


    PROFILER = lambda d: json.dumps({"profiler": "torch", "torch_profiler_dir": str(d), "torch_profiler_with_stack": False,
                                     "ignore_frontend": True})


    def stock_flags(model, port, *, tp=4, pp=1, seqs=20, spec=3, kv="fp8_e4m3", backend="FLASHINFER", prefix=True, mem=0.90,
                    batched=4096, max_len=MAX_LEN, extra=(), profile_dir=None, spec_json=None):
        f = [str(model), "--tensor-parallel-size", str(tp), "--kv-cache-dtype", kv, "--max-model-len", str(max_len),
             "--max-num-seqs", str(seqs), "--max-num-batched-tokens", str(batched), "--gpu-memory-utilization", str(mem),
             "--disable-custom-all-reduce", "--seed", "20261005", "--host", "127.0.0.1", "--port", str(port),
             "--served-model-name", "q27", "--attention-backend", backend, "--trust-remote-code", "--language-model-only"]
        if pp > 1:
            f += ["--pipeline-parallel-size", str(pp)]
        f += ["--enable-prefix-caching"] if prefix else ["--no-enable-prefix-caching"]
        if spec_json:
            f += ["--speculative-config", spec_json]
        elif spec:
            f += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": int(spec)})]
        if profile_dir:
            f += ["--profiler-config", PROFILER(profile_dir)]
        return f + list(extra)


    def stock_server(name, flags, port, env_extra=None, engine=None, gpus="0,1,2,3"):
        eng = engine or STOCK
        env = dict(eng.get("env") or STOCK["env"])
        env.update(env_extra or {})
        if "--profiler-config" in flags:
            env["VLLM_CUSTOM_SCOPES_FOR_PROFILING"] = "1"                    # forward ranges for the per-forward graph verdict
        if not FAKE:
            HELP[name] = L.help_check(eng["name"], list(eng["vllm"]), env, flags)
        return L.Server(name, list(eng["vllm"]) + list(flags), env, port, LOGS, gpus=gpus)
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 3 — distinct ARC prompts per stream (token ids)
    from tokenizers import Tokenizer
    TOK = Tokenizer.from_file(str(FP8 / "tokenizer.json"))
    _tasks = json.load(open(ARC))
    _ids = sorted(_tasks)


    def _grid(g):
        return "\n".join("".join(str(v) for v in row) for row in g)


    def _task_text(k):
        t = _tasks[k]
        parts = [f"## Task {k}"]
        for i, ex in enumerate(t["train"]):
            parts.append(f"Example {i + 1} input:\n{_grid(ex['input'])}\nExample {i + 1} output:\n{_grid(ex['output'])}")
        for ex in t["test"]:
            parts.append(f"Test input:\n{_grid(ex['input'])}")
        return "\n\n".join(parts)


    _HEAD = TOK.encode("<|im_start|>user\nFind the transformation rule of each ARC task below and solve the test inputs.\n\n",
                       add_special_tokens=False).ids
    _TAIL = TOK.encode("\n<|im_end|>\n<|im_start|>assistant\n<think>\n", add_special_tokens=False).ids
    for _t in ("<|im_start|>", "<|im_end|>", "<think>"):             # rev 3: the chat markers are single special tokens
        _id = TOK.token_to_id(_t)
        assert _id is not None and _id in (_HEAD + _TAIL), (_t, _id)
    _TASK_IDS = {k: TOK.encode(_task_text(k) + "\n\n", add_special_tokens=False).ids for k in _ids}


    def prompt(band, variant):
        """~band tokens of DISTINCT ARC tasks (stream `variant` starts at its own task offset), chat-formatted, thinking on."""
        body, i = [], (variant * 7) % len(_ids)
        need = band - len(_HEAD) - len(_TAIL)
        while len(body) < need:
            body += _TASK_IDS[_ids[i % len(_ids)]]
            i += 1
        return _HEAD + body[:need] + _TAIL


    BANDS = {"2k": 2048, "8k": 8192, "40k": 40000, "90k": 90000, "120k": 120000}
    NSTREAM = 24
    PROMPTS = {b: [prompt(n, v) for v in range(NSTREAM)] for b, n in BANDS.items()}
    MAX_NEW = {b: MAX_LEN - n - 512 for b, n in BANDS.items()}       # a stream never ends inside a window
    ORACLE_P = [PROMPTS["2k"][0], PROMPTS["2k"][1], PROMPTS["8k"][2]]
    LONG_P = [PROMPTS["40k"][3]]                                      # one long-context greedy row (40k)
    TF_P = {"40k": PROMPTS["40k"][5], "90k": PROMPTS["90k"][5]}        # teacher-forced long-context rows (NVFP4 vs FP8 KV)
    R["prompts"] = {b: {"n": len(p), "len": len(p[0]), "sha": hashlib.sha256(json.dumps(p[:4]).encode()).hexdigest()[:12],
                        "distinct_first_task": len({tuple(x[len(_HEAD):len(_HEAD) + 50]) for x in p})} for b, p in PROMPTS.items()}
    log("prompts", json.dumps(R["prompts"]))
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 4 — the step runner (budget, GPU gate, fail-closed)
    WIN = {"SMOKE": 30.0, "FULL": 120.0}[MODE]                         # 120 s FULL windows (noise)
    ROWS = {}            # oracle greedy rows by key (same weights)
    TF = {}              # teacher-forced rows by key
    NTOK = 128 if MODE == "FULL" else 64


    def step(sec, name, est_min, fn):
        """One measured step: skipped when the budget cannot hold it, recorded on any exception, never PASS by default."""
        rec = R["sections"].setdefault(sec, {}).setdefault(name, {"status": "NOT_RUN"})
        R["order"].append(f"{sec}/{name}")
        if ABORT["why"]:
            rec.update(status="SKIPPED_ABORT", why=ABORT["why"])
            save()
            return rec
        if left() < est_min * 60 + 300:
            rec.update(status="SKIPPED_BUDGET", left_min=round(left() / 60, 1), est_min=est_min)
            save()
            return rec
        t = time.time()
        log(">>>", sec, name, f"(est {est_min} min, left {left() / 60:.0f} min)")
        try:
            out = fn() or {}
            rec.update(out)
            if rec.get("status") in (None, "NOT_RUN"):
                rec["status"] = "DONE"
        except Exception as e:
            rec.update(status="ERROR", error=f"{type(e).__name__}: {e}"[:800], tb=traceback.format_exc()[-2500:])
        rec["sec"] = round(time.time() - t, 1)
        if ABORT["why"] and rec["status"] == "DONE":
            rec["status"] = "PARTIAL_HARD_STOP"
        log("<<<", sec, name, rec.get("status"), f"{rec['sec']:.0f}s")
        save()
        return rec


    def with_servers(servers, timeout, body):
        """Start, wait ready, record facts + memory, run body(servers), always kill; then the GPU-release gate on these
        servers' GPUs only; abort the lab if they stay held."""
        out = {"servers": {}}
        timeout = max(60, min(timeout, left() - 120))
        try:
            for s in servers:
                s.start()
            for s in servers:
                ok, why = s.wait_ready(timeout)
                out["servers"][s.name] = {"ready": ok, "why": why, **s.facts(), "argv_tail": s.argv[-44:], "gpus": s.gpus,
                                          "help_check": HELP.get(s.name)}
                if not ok:
                    out["status"] = "LAUNCH_FAILED"
                    out["servers"][s.name]["log_tail"] = s.log()[-4000:]
                    return out
            mem0 = L.gpu_mem()
            for s in servers:
                out["servers"][s.name].update(gpu_mem_after_ready=mem0, host_rss_mb=s.rss_mb())
            try:                             # rev 3: an exception in the body keeps everything measured so far
                out.update(body(servers) or {})
            except Exception as e:
                out.update(status="BODY_ERROR", body_error=f"{type(e).__name__}: {e}"[:800], body_tb=traceback.format_exc()[-2000:])
            for s in servers:
                out["servers"][s.name].update(s.facts(), gpu_mem_end=L.gpu_mem(), host_rss_mb_end=s.rss_mb(),
                                              alive_after_body=s.alive(), rc=None if s.alive() else s.proc.returncode)
                if not s.alive():
                    out["status"] = "SERVER_DIED"
                    out["servers"][s.name]["log_tail"] = s.log()[-4000:]
            return out
        finally:
            for s in servers:
                s.kill()
            gpus = ",".join(sorted({g for s in servers for g in s.gpus.split(",")}))
            rel = L.wait_gpu_release(180, fake=bool(FAKE), gpus=gpus)
            out["gpu_released"] = rel
            if rel is False:
                ABORT["why"] = f"GPUs {gpus} still held after {[s.name for s in servers]}"
                out["status"] = "FAIL_GPU_RELEASE"


    def oracle(port, key=None, refs=()):
        """Greedy rows (2k, 2k, 8k) + one 40k row; saved under `key` (a target-only oracle) or compared with ROWS[ref]."""
        rows = L.greedy(port, ORACLE_P + LONG_P, n_tok=NTOK, timeout=max(120, min(900, left() / 6)))
        if key:
            ROWS[key] = rows
        res = {"garbage": [L.degenerate(r["text"]) for r in rows], "errors": [r["error"] for r in rows],
               "text_head": [r["text"][:120] for r in rows]}
        for ref in refs:
            res["vs_" + ref] = L.compare(ROWS.get(ref), rows) if ROWS.get(ref) else {"status": "INCONCLUSIVE_NO_ORACLE"}
        return res


    def tforced(port, key=None, ref=None):
        """Teacher-forced 40k / 90k prompt logprobs (last 256 positions); saved under `key` or compared with TF[ref]."""
        rows = {b: L.teacher_forced(port, ids) for b, ids in TF_P.items()}
        if key:
            TF[key] = rows
        if ref:
            return {b: L.compare_tf((TF.get(ref) or {}).get(b), rows[b]) for b in rows}
        return {b: {"n": (rows[b] or {}).get("n"), "error": (rows[b] or {}).get("error")} for b in rows}


    def levels(port, band, lv, **kw):
        out = {}
        for n in lv:
            if ABORT["why"] or left() < WIN + 300:
                out[f"{band}@{n}"] = {"status": "SKIPPED_BUDGET"}
                continue
            try:                             # rev 3: one failing level never loses the others
                out[f"{band}@{n}"] = L.steady(port, PROMPTS[band], n, secs=WIN, max_new=MAX_NEW[band], **kw)
            except Exception as e:
                out[f"{band}@{n}"] = {"status": "ERROR", "error": f"{type(e).__name__}: {e}"[:300]}
            save()
        return out
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 5 — the plan (dependency order; report = user numbering)
    import concurrent.futures as CF
    SM = MODE == "SMOKE"
    DO_A = (not SM) or PART in ("A", "ALL")          # Merlin, P5, P4
    DO_B = (not SM) or PART in ("B", "ALL")          # stock vLLM + EAGLE + PP4
    CAP_TP1 = [1, 2, 5, 10, 15, 20]                                  # seqs 4 x (1 + MTP 4) = 20 (V2 runner rounds to x5)
    CAP_TP4 = [1, 2, 4, 5, 8, 10, 15, 20, 25, 30, 35, 40, 60, 80, 100, 120]   # seqs 24 x 5 = 120
    KV_TP1, KV_TP4 = 8055029760, 16301686784                         # Merlin preflight's L4 (22.5 GiB) pins
    P4_FUT = None
    TP1_CFG = {"seqs": 4, "capture": CAP_TP1}


    def kit(name):
        return str(KITS / name)


    # ---- #2 host-staged all-reduce: all four GPUs free, before any server
    def s2():
        out = str(OUT / "p5_bench.json")
        Path(out).unlink(missing_ok=True)
        cmd = [STOCK["py"], kit("bench_host_ar.py"), "--so", kit("p5_host_ar.so"), "--out", out,
               "--timeout-s", "900" if SM else "3000"] + (["--quick"] if SM else [])
        try:
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=1000 if SM else 3300, env=STOCK["env"])
            rc, tail = p.returncode, (p.stdout + p.stderr)[-2000:]
        except subprocess.TimeoutExpired as e:
            rc, tail = -9, str(e)[-500:]
        res = json.loads(Path(out).read_text()) if Path(out).is_file() else {}
        return {"rc": rc, "summary": res.get("summary"), "tail": tail,
                "status": "DONE" if rc == 0 else ("CHECK_FAILED" if rc == 1 else "BENCH_ERROR")}


    def s4_thread():
        out = str(OUT / "p4_bench.json")
        Path(out).unlink(missing_ok=True)
        cmd = [STOCK["py"], kit("bench_fp8_gemm.py"), "--so", kit("p4_sm89_fp8.so"), "--out", out, "--gpu", "3",
               "--weights", "auto", "--model-dir", str(FP8)] + (["--m", "1,16,80", "--shapes", "gate_up_proj,down_proj"] if SM else [])
        try:
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=1500 if SM else 3600, env=STOCK["env"],
                               start_new_session=True)
            rc, tail = p.returncode, (p.stdout + p.stderr)[-2000:]
        except subprocess.TimeoutExpired as e:
            rc, tail = -9, str(e)[-500:]
        res = json.loads(Path(out).read_text()) if Path(out).is_file() else {}
        return {"rc": rc, "summary_per_m": res.get("summary_per_m"), "gates": res.get("gates") or res.get("gate"),
                "tail": tail, "status": "DONE" if rc == 0 else "BENCH_ERROR"}


    def join_p4():
        """P4 holds GPU 3; join it (bounded) before any step that needs GPU 3."""
        global P4_FUT
        if P4_FUT is None:
            return
        try:
            res = P4_FUT.result(timeout=max(60, min(3600, left() - 600)))
        except Exception as e:
            res = {"status": "ERROR", "error": repr(e)[:300]}
        R["sections"].setdefault("4_sm89_fp8_gemm", {})["p4_bench_gpu3"] = res
        P4_FUT = None
        L.wait_gpu_release(180, fake=bool(FAKE), gpus="3")
        save()


    # ---- #1 Merlin TP1 (GPU 0) with #4 P4 on GPU 3 meanwhile; memory fallback; replicas; Merlin TP4
    def _tp1_body(ss):
        port = ss[0].port
        o = {"probes": L.chat_probes(port), "oracle": oracle(port, key="merlin")}
        o["steady"] = levels(port, "40k", [2, 4] if SM else [2, 3, 4])
        o["steady"].update(levels(port, "90k", [1, 2]))
        o["steady"].update(levels(port, "120k", [1] if SM else [1, 2]))
        return o


    def s1_tp1():
        o = with_servers([merlin_server("merlin_tp1", "27b-tp1-sm89-24g", "0", 8100, KV_TP1, **TP1_CFG)], 2400, _tp1_body)
        if o.get("status") == "LAUNCH_FAILED":           # rev 3: memory edge -> Merlin's own TP1 profile values (seqs 2)
            TP1_CFG.update(seqs=None, capture=None)
            o2 = with_servers([merlin_server("merlin_tp1_fallback", "27b-tp1-sm89-24g", "0", 8100, KV_TP1)], 2400, _tp1_body)
            o2["first_attempt"] = {k: o[k] for k in ("status", "servers") if k in o}
            o2["config"] = "profile defaults (seqs 2, capture [1,2,5,10])"
            return o2
        o["config"] = "seqs 4, capture " + str(CAP_TP1)
        return o


    def _tp1_level(band, default):
        rec = R["sections"]["1_tp1x4_pp4"].get("a_merlin_tp1_gpu0") or {}
        st = rec.get("steady") or {}
        ok = [int(k.split("@")[1]) for k, v in st.items() if k.startswith(band + "@") and v.get("status") in ("STEADY", "CAPACITY_LIMITED")]
        return max(ok) if ok else default


    def s1_rep(n):
        def f():
            lv40, lv90 = _tp1_level("40k", 2), _tp1_level("90k", 1)
            ss = [merlin_server(f"merlin_tp1_rep{i}", "27b-tp1-sm89-24g", str(i), 8100 + i, KV_TP1, **TP1_CFG) for i in range(n)]

            def body(ss):
                o = {"per_replica_level": {"40k": lv40, "90k": lv90},
                     "replica_greedy": {s.name: L.compare(ROWS.get("merlin")[:1], L.greedy(s.port, ORACLE_P[:1], n_tok=NTOK))
                                        if ROWS.get("merlin") else {"status": "INCONCLUSIVE_NO_ORACLE"} for s in ss}}
                o["40k"] = L.steady_multi([(s.port, PROMPTS["40k"][i * 6:], lv40) for i, s in enumerate(ss)], secs=WIN,
                                          max_new=MAX_NEW["40k"])
                if not SM:
                    o["90k"] = L.steady_multi([(s.port, PROMPTS["90k"][i * 6:], lv90) for i, s in enumerate(ss)], secs=WIN,
                                              max_new=MAX_NEW["90k"])
                return o
            return with_servers(ss, 2400, body)
        return f


    def s1_tp4():
        def body(ss):
            port = ss[0].port
            o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("merlin",))}
            o["steady"] = levels(port, "40k", [8, 16] if SM else [8, 16, 24])
            o["steady"].update(levels(port, "90k", [8] if SM else [8, 12, 16]))
            o["steady"].update(levels(port, "120k", [8] if SM else [8, 12]))
            return o
        o = with_servers([merlin_server("merlin_tp4", "27b-tp4-sm89-24g", "0,1,2,3", 8100, KV_TP4, seqs=24, capture=CAP_TP4)], 2400, body)
        if o.get("status") == "LAUNCH_FAILED":           # the profile's own capture list (up to 40)
            o2 = with_servers([merlin_server("merlin_tp4_fallback", "27b-tp4-sm89-24g", "0,1,2,3", 8100, KV_TP4, seqs=24)], 2400, body)
            o2["first_attempt"] = {k: o[k] for k in ("status", "servers") if k in o}
            return o2
        return o


    # ---- #6 MTP OFF / K1 / K3 (FP8, stock vLLM 0.32, seqs 20); K3 = G0 of #3 (FlashInfer, PIECEWISE pinned), profiled
    def s6(depth, profile=False, tag=None):
        def f():
            pdir = TMP / f"e23_profile_mtp{depth}_{tag or 'main'}" if profile else None
            extra = ["--compilation-config", json.dumps({"cudagraph_mode": "PIECEWISE"})] if depth == 3 else []
            srv = stock_server(tag or f"fp8_mtp{depth}", stock_flags(FP8, 8200, spec=depth, extra=extra, profile_dir=pdir), 8200)

            def body(ss):
                port = ss[0].port
                o = {"probes": L.chat_probes(port),
                     "oracle": oracle(port, key=("fp8_mtp0" if depth == 0 else ("fp8_g0" if depth == 3 else None)) if not tag else None,
                                      refs=() if depth == 0 and not tag else ("fp8_mtp0",))}
                if depth == 0 and not tag:
                    o["teacher_forced"] = tforced(port, key="fp8_mtp0")
                o["steady"] = levels(port, "40k", [16] if SM else [8, 12, 16, 20])
                if not SM and depth in (0, 3):
                    o["steady"].update(levels(port, "90k", [8, 12, 20]))
                    o["steady"].update(levels(port, "120k", [8, 12]))
                if profile:
                    o["profile"] = L.profiled_load(port, PROMPTS["40k"], 16, pdir, max_new=MAX_NEW["40k"])
                    shutil.rmtree(pdir, ignore_errors=True)
                return o
            return with_servers([srv], 1800, body)
        return f


    def _site():
        return subprocess.run([STOCK["py"], "-c", "import sysconfig;print(sysconfig.get_paths()['purelib'])"],
                              capture_output=True, text=True).stdout.strip()


    def _patch(reverse=False):
        """rev 3: forward = back up flashinfer.py, dry-run, apply (restore the backup on a failed apply); reverse = restore
        the backup by copy and verify its sha (not patch -R)."""
        target = Path(_site()) / "vllm/v1/attention/backends/flashinfer.py"
        bak = TMP / "flashinfer.py.e23_orig"
        if reverse:
            if not bak.is_file():
                return {"applied": False, "error": "no backup"}
            shutil.copy2(bak, target)
            ok = hashlib.sha256(target.read_bytes()).hexdigest() == hashlib.sha256(bak.read_bytes()).hexdigest()
            return {"applied": ok, "restored_sha": hashlib.sha256(target.read_bytes()).hexdigest()[:16]}
        shutil.copy2(target, bak)
        diff = P60532.split("diff --git a/vllm/v1/attention/backends/flashinfer.py")[1]
        Path(TMP / "pr60532_vllm_only.diff").write_text("diff --git a/vllm/v1/attention/backends/flashinfer.py" + diff.split("\ndiff --git")[0] + "\n")
        base = ["patch", "-p1", "-d", _site(), "--batch", "--forward", "--no-backup-if-mismatch", "-i", str(TMP / "pr60532_vllm_only.diff")]
        dry = subprocess.run(base + ["--dry-run"], capture_output=True, text=True, stdin=subprocess.DEVNULL, timeout=120)
        if dry.returncode:
            return {"rc": dry.returncode, "dry_run": (dry.stdout + dry.stderr)[-600:], "applied": False}
        p = subprocess.run(base, capture_output=True, text=True, stdin=subprocess.DEVNULL, timeout=120)
        if p.returncode:
            shutil.copy2(bak, target)
        return {"rc": p.returncode, "out": (p.stdout + p.stderr)[-600:], "applied": p.returncode == 0,
                "orig_sha": hashlib.sha256(bak.read_bytes()).hexdigest()[:16]}


    def s3(tag, backend, graph, patch=False):
        def f():
            pre = _patch() if patch and not FAKE else None
            if pre and not pre["applied"]:
                return {"status": "PATCH_FAILED", "patch": pre}
            try:
                pdir = TMP / f"e23_profile_{tag}"
                extra = ["--compilation-config", json.dumps({"cudagraph_mode": graph})] if graph else []
                srv = stock_server(f"fp8_{tag}", stock_flags(FP8, 8200, backend=backend, extra=extra, profile_dir=pdir), 8200)

                def body(ss):
                    port = ss[0].port
                    o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("fp8_mtp0", "fp8_g0")), "patch": pre}
                    o["steady"] = levels(port, "40k", [16] if SM else [16, 20])
                    if not SM:
                        o["steady"].update(levels(port, "90k", [12]))
                    o["profile"] = L.profiled_load(port, PROMPTS["40k"], 16, pdir, max_new=MAX_NEW["40k"])
                    shutil.rmtree(pdir, ignore_errors=True)
                    return o
                return with_servers([srv], 1800, body)
            finally:
                if pre and pre.get("applied"):
                    rv = _patch(reverse=True)
                    R["sections"].setdefault("3_full_graph", {})[f"{tag}_patch_revert"] = rv
                    if not rv.get("applied"):
                        ABORT["why"] = "PR #60532 revert failed: the stock venv is no longer pristine"
        return f


    # ---- #5 NVFP4 KV: greedy + teacher-forced correctness vs FP8-KV target-only, 90k / 120k census, + MTP3 (not on NO_GO)
    def s5(spec, tag=None):
        def f():
            prior = (R["sections"].get("5_nvfp4_kv") or {}).get("nvfp4_target_only") or {}
            if spec and prior.get("status") in ("NO_GO_CORRUPT", "LAUNCH_FAILED", "ERROR", "FAIL_GPU_RELEASE", "SKIPPED_BUDGET"):
                return {"status": "SKIPPED_TARGET_ONLY_NO_GO", "target_only_status": prior.get("status")}
            srv = stock_server(tag or f"nvfp4_mtp{spec}", stock_flags(FP8, 8200, spec=spec, kv="nvfp4", seqs=24), 8200)

            def body(ss):
                port = ss[0].port
                o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("fp8_mtp0",) if spec == 0 else ("fp8_g0", "fp8_mtp0"))}
                if spec == 0:
                    o["teacher_forced_vs_fp8kv"] = tforced(port, ref="fp8_mtp0")
                    if (o["oracle"].get("vs_fp8_mtp0") or {}).get("status") == "FAIL_HIGH_MARGIN":
                        o["status"] = "NO_GO_CORRUPT"
                        return o
                o["steady"] = levels(port, "90k", [12] if SM else [12, 16, 20, 24])
                o["steady"].update(levels(port, "120k", [12] if SM else [12, 16, 20]))
                if not SM:
                    o["steady"].update(levels(port, "40k", [16, 20]))
                return o
            return with_servers([srv], 1800, body)
        return f


    # ---- #4 INT4 control (stock vLLM, RedHat W4A16, MTP3, s20): the INT4 path against Merlin's low-M INT4 engine
    def s4_int4():
        srv = stock_server("int4_mtp3_s20", stock_flags(INT4, 8200, spec=3, seqs=20), 8200)

        def body(ss):
            port = ss[0].port
            o = {"probes": L.chat_probes(port), "oracle": oracle(port, key="int4_mtp3")}
            o["steady"] = levels(port, "40k", [8, 16, 20])
            o["steady"].update(levels(port, "90k", [12]))
            return o
        return with_servers([srv], 1800, body)


    # ---- #1e PP4 branch (stock vLLM, INT4 RedHat): eager first (#63505's only reported pass), then compiled; stop on garbage
    def s1_pp4(eager):
        def f():
            extra = ["--enforce-eager"] if eager else []
            srv = stock_server("int4_pp4_eager" if eager else "int4_pp4_compiled",
                               stock_flags(INT4, 8200, tp=1, pp=4, spec=0, seqs=16, extra=extra), 8200)

            def body(ss):
                port = ss[0].port
                o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("int4_mtp3",)),
                     "note": "PP4 uses stock vLLM + RedHat INT4 (Merlin's fork has no PP profile); #63505 open on 0.32"}
                if not o["probes"]["ok"] or any(o["oracle"]["garbage"]) or any(o["oracle"]["errors"]):
                    o["status"] = "NO_GO_GARBAGE_OR_ERROR"
                    return o
                o["steady"] = levels(port, "40k", [8] if SM else [8, 16])
                return o
            return with_servers([srv], 1800, body)
        return f


    # ---- EAGLE-4 feasibility: a BF16-KV FP8-weight target-only oracle first (same KV dtype as tree verify), then EAGLE
    def s7_kvauto_oracle():
        srv = stock_server("fp8_kvauto_mtp0", stock_flags(FP8, 8300, spec=0, kv="auto", seqs=16), 8300)
        return with_servers([srv], 1800, lambda ss: {"oracle": oracle(ss[0].port, key="fp8_kvauto_mtp0")})


    def s7_eagle():
        eng = eagle_install()
        if eng is None:
            return {"status": "SKIPPED_REQUIRED_ARTIFACT", "why": "EAGLE env / drafter not attached (or fake mode)"}
        spec = json.dumps({"method": "eagle_harness", "model": str(EAGLE_DRAFT),
                           "method_config": {"mode": "static", "node_top_k": 4, "tree_depth": 14, "max_verify_width": 16}})
        fl = stock_flags(FP8, 8300, seqs=16, kv="auto", backend="CUSTOM", prefix=False, spec_json=spec)
        srv = stock_server("eagle4_fp8", fl, 8300, engine=eng)

        def body(ss):
            port = ss[0].port
            o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("fp8_kvauto_mtp0", "fp8_mtp0"))}
            o["steady"] = levels(port, "8k", [8, 16])
            o["steady"].update(levels(port, "40k", [8]))
            return o
        return with_servers([srv], 2400, body)


    PREF = []
    if DO_A and MERLIN:
        PREF.append(prefetch_bg([MERLIN], "merlin"))
    if DO_B:
        PREF.append(prefetch_bg([FP8] + ([INT4] if INT4 and not SM else []), "fp8_int4"))

    # ---- run order. SMOKE A: P5, Merlin TP1 (+P4), Merlin TP4. SMOKE B: FP8 mtp0 -> mtp3/G0 -> G3 -> NVFP4 -> EAGLE -> PP4.
    # FULL: P5, Merlin block, section 6, section 3, section 5, INT4 control, A/A repeats (time left), PP4 last (#63505).
    if DO_A:
        step("2_host_allreduce", "p5_bench", 12 if SM else 30, s2)
        P4_FUT = CF.ThreadPoolExecutor(1).submit(s4_thread)
        step("1_tp1x4_pp4", "a_merlin_tp1_gpu0", 45 if SM else 75, s1_tp1)
        join_p4()
        if not SM:
            step("1_tp1x4_pp4", "b_merlin_tp1_x2", 30, s1_rep(2))
            step("1_tp1x4_pp4", "c_merlin_tp1_x4", 35, s1_rep(4))
        step("1_tp1x4_pp4", "d_merlin_tp4", 40 if SM else 60, s1_tp4)
    if DO_B:
        step("6_mtp_depth", "mtp0_target_only", 20 if SM else 50, s6(0))
        if not SM:
            step("6_mtp_depth", "mtp1", 35, s6(1))
        step("6_mtp_depth", "mtp3_eq_G0_piecewise", 25 if SM else 60, s6(3, profile=True))
        if not SM:
            step("3_full_graph", "G1_triton_full_and_piecewise", 35, s3("G1", "TRITON_ATTN", "FULL_AND_PIECEWISE"))
            step("3_full_graph", "G2_triton_full_decode_only", 35, s3("G2", "TRITON_ATTN", "FULL_DECODE_ONLY"))
        step("3_full_graph", "G3_flashinfer_pr60532", 20 if SM else 40, s3("G3", "FLASHINFER", "FULL_AND_PIECEWISE", patch=True))
        step("5_nvfp4_kv", "nvfp4_target_only", 25 if SM else 55, s5(0))
        if not SM:
            step("5_nvfp4_kv", "nvfp4_mtp3", 45, s5(3))
            step("4_sm89_fp8_gemm", "int4_mtp3_s20_control", 35, s4_int4)
        if SM or os.environ.get("E23_EAGLE_FULL") == "1":
            step("7_eagle4", "fp8_kvauto_target_only_oracle", 15, s7_kvauto_oracle)
            step("7_eagle4", "eagle4_fp8_tp4", 30, s7_eagle)
    if not SM:      # A/A repeats of the decisive comparisons while 70 min stay reserved for PP4
        for i in range(1, 4):
            if left() < (70 + 50) * 60:
                break
            step("8_aa_repeats", f"mtp3_control_rep{i}", 50, s6(3, tag=f"fp8_mtp3_rep{i}"))
            step("8_aa_repeats", f"merlin_tp1_x4_rep{i}", 35, s1_rep(4))
            step("8_aa_repeats", f"nvfp4_mtp3_rep{i}", 45, s5(3, tag=f"nvfp4_mtp3_rep{i}"))
    if DO_B:
        step("1_tp1x4_pp4", "e_pp4_int4_eager", 20 if SM else 30, s1_pp4(True))
        step("1_tp1x4_pp4", "f_pp4_int4_compiled", 20 if SM else 30, s1_pp4(False))


In [ ]:
if not RERUN:
    # ==================================================================== CELL 6 — report
    def _flat(sec):
        rows = []
        for name, rec in (R["sections"].get(sec) or {}).items():
            if not isinstance(rec, dict):
                continue
            for k, v in (rec.get("steady") or {}).items():
                rows.append([sec, name, k, v.get("status"), v.get("tok_s"), v.get("running_mean"), v.get("mean_acceptance_length"),
                             v.get("step_ms"), v.get("kv_usage_max"), v.get("preemptions")])
            for k in ("40k", "90k"):
                if isinstance(rec.get(k), dict) and "replicas" in rec[k]:
                    rows.append([sec, name, k + " aggregate", rec[k]["status"], rec[k].get("tok_s"), rec[k].get("running_mean"),
                                 None, None, None, None])
        return rows


    lines = ["| section | step | window | status | tok/s | running | accept len | step ms | kv max | preempt |",
             "|---|---|---|---|---|---|---|---|---|---|"]
    for sec in sorted(R["sections"]):
        for r_ in _flat(sec):
            lines.append("| " + " | ".join("-" if x is None else str(x) for x in r_) + " |")
    st = ["| section | step | status | windows steady/all | oracle | probes | KV tokens | ready s | graph capture | trace verdict |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    for sec in sorted(R["sections"]):
        for name, rec in R["sections"][sec].items():
            if not isinstance(rec, dict):
                continue
            o = rec.get("oracle") or {}
            vs = ", ".join(f"{k[3:]}={v.get('status')}" for k, v in o.items() if k.startswith("vs_") and isinstance(v, dict))
            sv = next(iter((rec.get("servers") or {}).values()), {}) if rec.get("servers") else {}
            tr_ = (rec.get("profile") or {}).get("trace") or {}
            gv = tr_.get("graph_verdict") or tr_.get("error") or "-"
            if tr_.get("graph_verdict"):
                gv += f" (g/fw {tr_.get('per_forward_median_graph_launches')}, k/fw {tr_.get('per_forward_median_kernel_launches')})"
            wins = list((rec.get("steady") or {}).values())
            ws = f"{sum(1 for w in wins if w.get('status') in ('STEADY', 'CAPACITY_LIMITED'))}/{len(wins)}" if wins else "-"
            st.append(f"| {sec} | {name} | {rec.get('status')} | {ws} | {vs or ('saved' if o else '-')} | "
                      f"{(rec.get('probes') or {}).get('ok', '-')} | {sv.get('kv_tokens', '-')} | {sv.get('ready_sec', '-')} | "
                      f"{','.join(sv.get('graph_capture_modes') or []) or '-'} | {gv} |")
    R["help_check"] = HELP
    R["report_md"] = "\n".join(st) + "\n\n" + "\n".join(lines)
    R["minutes"] = round((time.time() - T_START) / 60, 1)
    save()
    print(R["report_md"])
    print("E23", MODE, PART, "done in", R["minutes"], "min | abort:", ABORT["why"])
